In [ ]:
"""
PHASE 0: HYPOTHESIS VALIDATION
Greedy vs. sampled decode on the best checkpoint.
If greedy >> sampled: problem is sampling strategy.
If greedy ≈ sampled: problem is the model weights / training objective.
"""

import json, re, math, torch
from pathlib import Path
import statistics

# Load best prose checkpoint
ck = torch.load('runs/best_lm.pt', map_location='cpu', weights_only=True)
print(f'Checkpoint: kind={ck["kind"]}, slots={ck.get("slots")}, validation_bpb={ck.get("validation_bpb"):.4f}')

from model import LM
model = LM(ck['kind'], ck.get('slots', 8), read_window=ck.get('read_window')).to(device)
model.load_state_dict(ck['state_dict'])
model.eval()

# Test prompts
TEST_PROMPTS = [
    'The expedition reached the river at dawn. ',
    'Gauthier and Knoote married on May 22 , 1911 . ',
    'Income from open days at the mill came to £ 151 . ',
    'There is disagreement among music scholars as to its importance . ',
]

def gen_greedy(prompt, max_bytes=160):
    tokens = list(prompt.encode('utf8'))
    chunk = model.s.chunk
    state = None
    start = 0
    with torch.inference_mode():
        for step in range(max_bytes):
            while len(tokens) - start > chunk:
                ids = torch.tensor([tokens[start:start+chunk]], device=device, dtype=torch.long)
                _, state = model.stream_chunk(ids, state)
                start += chunk
            ids = torch.tensor([tokens[start:]], device=device, dtype=torch.long)
            logits, pending = model.stream_chunk(ids, state)
            byte = logits[0, -1].argmax().item()
            tokens.append(byte)
            if byte == 10: break
            if len(tokens) - start > chunk:
                state = pending
    return bytes(tokens[len(prompt.encode()):]).decode('utf8', errors='replace')

def gen_sampled(prompt, max_bytes=160, temp=0.75, top_p=0.9):
    tokens = list(prompt.encode('utf8'))
    chunk = model.s.chunk
    state = None
    start = 0
    with torch.inference_mode():
        for step in range(max_bytes):
            while len(tokens) - start > chunk:
                ids = torch.tensor([tokens[start:start+chunk]], device=device, dtype=torch.long)
                _, state = model.stream_chunk(ids, state)
                start += chunk
            ids = torch.tensor([tokens[start:]], device=device, dtype=torch.long)
            logits, pending = model.stream_chunk(ids, state)
            logits = logits[0, -1].float() / temp
            probs = torch.softmax(logits, dim=0)
            sorted_p, sorted_i = torch.sort(probs, descending=True)
            cumsum = torch.cumsum(sorted_p, dim=0)
            to_remove = cumsum > top_p
            to_remove[0] = False
            probs[sorted_i[to_remove]] = 0
            probs = probs / probs.sum()
            byte = torch.multinomial(probs, 1).item()
            tokens.append(byte)
            if byte == 10: break
            if len(tokens) - start > chunk:
                state = pending
    return bytes(tokens[len(prompt.encode()):]).decode('utf8', errors='replace')

def score(text):
    t = text.replace('@-@', '-').replace('@,@', ',')
    words = len(re.findall(r"[A-Za-z']+", t))
    sents = [s for s in re.split(r'(?<=[.!?])\s+', t.strip()) if s]
    complete = sum(1 for s in sents if re.search(r'[.!?]$', s))
    dangling = bool(re.search(r'\b(and|or|of|in|on|at|to|with|the|a|an|but)$', t, re.I))
    return dict(text=t[:90], words=words, sents=len(sents), complete=complete, dangling=dangling)

print('\n' + '='*70)
print('PHASE 0: GREEDY vs SAMPLED')
print('='*70 + '\n')

results = {}
for strat, gen_fn in [('greedy', gen_greedy), ('sampled', gen_sampled)]:
    results[strat] = []
    for prompt in TEST_PROMPTS:
        text = gen_fn(prompt)
        results[strat].append(score(text))

for i, prompt in enumerate(TEST_PROMPTS):
    print(f'Prompt: {prompt[:60]!r}')
    for strat in ('greedy', 'sampled'):
        r = results[strat][i]
        print(f'  {strat:8s}: {r["words"]:3d}w {r["complete"]}/{r["sents"]}s dangling={r["dangling"]:5} | {r["text"]!r}')
    print()

# Summary
for strat in ('greedy', 'sampled'):
    rr = results[strat]
    comp_rate = sum(r['complete'] for r in rr) / max(1, sum(r['sents'] for r in rr))
    print(f'{strat.upper():8s} complete-sentence rate: {100*comp_rate:.0f}%')

print('\n=> If greedy >> sampled: switch to greedy decoding (problem is sampling).')
print('=> If greedy ≈ sampled: retrain with better prose objective (problem is weights).')

# Train HGRT-10M for fact-to-answer recall, sustained prose and conversation

In Colab, open this notebook directly from GitHub, choose **Runtime → Change runtime type → A100 GPU**, and click **Runtime → Run all**. Every required module (`model.py`, `dialogue.py`, `fact_switch.py`) is embedded verbatim inside this notebook so it runs standalone with no extra files on disk. By default the notebook resumes the newest matching WikiText run or `best_lm.pt` it can find in Drive or the local `runs/` folder; set `AUTO_RESUME = False` for a fresh start, `RESUME_FROM` for a specific `latest.pt`, or `INITIAL_WEIGHTS` to start from saved `C1-S8` weights.

**Research pipeline (in priority order):**
1. **WikiText-103 prose pretraining / resume (`TRAIN_MINUTES`)** — pinned raw Parquet shards ([Dataset card](https://huggingface.co/datasets/Salesforce/wikitext) · [Merity et al.](https://arxiv.org/abs/1609.07843)), with matched vs. mismatched paragraph-continuation scoring.
2. **Counterfactual fact-to-answer learning (`FACT_SWITCH_MINUTES`)** — two-stage training directly targeting fact-to-answer switching on strictly disjoint OOV entities and novel nonce words: Stage 1 (`plain_nll`) locks in multi-byte OOV copying; Stage 2 (`contrastive` on 2×2 value-swap + question-swap quads) cancels both static word priors and Sentence-1 positional shortcuts. Selects checkpoints by counterfactual pair accuracy and greedy exact match rather than average BPB, and logs causal Tree ablations (`on` / `off` / `shuffle` / `counterfactual_tree_swap`). Set `RUN_ARCHITECTURE_SHOWDOWN = True` on an A100 to run a matched 4-arm head-to-head (`C1 L7-only`, `C1 L3+L7`, native `C1 rw=64` Tree-only, and same-size 10M Transformer `A0`).
3. **Rebalanced grounded QA, sustained prose & service chat (`DIALOGUE_MINUTES`)** — separates grounded QA (`Passage:/Question:/Answer:`) from service chat (`User:/Assistant:`) to stop domain leakage, auto-tunes CUDA batch size even when `TRAIN_MINUTES = 0`, rebalances QA + counterfactual quads + multi-sentence `prose_long` continuations, and preserves counterfactual fact-switching during checkpoint selection.

In [ ]:
#@title Run settings
from pathlib import Path

TRAIN_MINUTES = 20             # Additional WikiText prose training minutes per invocation (0 skips)
CONTEXT = 512                  # UTF-8 bytes per prose sequence
BATCH_BYTES = 8192             # Starting batch budget, in input bytes
AUTO_BATCH = True              # Benchmark larger CUDA batches separately for each training phase
MAX_BATCH_BYTES = 131072       # Upper budget for the automatic search
GPU_MEMORY_FRACTION = .85      # Fraction of available GPU memory the search may use
LR = 0.0005
SEED = 42
DEVICE = 'cuda'
TRY_FUSED = True               # Attempt the optional FLA gated-delta kernel
SAVE_TO_DRIVE = True
AUTO_RESUME = True             # Continue the newest matching prose run when available
INITIAL_WEIGHTS = ''           # Optional best_lm.pt or other C1-S8 weights for a new optimizer
RESUME_FROM = ''               # Optional latest.pt from this notebook
LOCAL_PARQUET_DIR = ''         # Optional offline directory with the four official shard names
OUTPUT_ROOT = ''               # Optional explicit output directory parent
MAX_UPDATES = None             # Optional additional update limit for a short diagnostic
VALID_WINDOWS = 128
TEST_WINDOWS = 256
PROBE_PAIRS = 64               # Per task, each with two matched variants
RUN_REASONING = False          # Optional original probes, after prose evaluation
TASK_MINUTES = 0               # Optional additional practice when RUN_REASONING is True

# Priority #1: Counterfactual fact-to-answer learning & optional A100 architecture showdown
FACT_SWITCH_MINUTES = 6        # Two-stage 2x2 quad fact-switching stage (0 evaluates zero-shot only)
FACT_SWITCH_LR = 0.0005
FACT_SWITCH_EVAL_PAIRS = 16    # Held-out OOV pairs per diagnostic split
RUN_ARCHITECTURE_SHOWDOWN = False  # Optional A100 showdown: C1(L7), C1(L3+L7), C1(rw=64 Tree), A0(10M)
SHOWDOWN_STEPS = 1200          # Optimizer steps per showdown arm (batch=4 quads/step on CUDA, 1 on CPU)

# Priority #2: Rebalanced grounded QA, sustained prose and conversation
DIALOGUE_MINUTES = 12          # Additional time for QA, sustained prose, chat and WikiText replay
DIALOGUE_CONTEXT = 1536        # Complete passage + turn history + target, in UTF-8 bytes
DIALOGUE_LR = .00015
DIALOGUE_EVAL_ROWS = 32        # Fixed rows per task and split
QA_PROMPT_STYLE = 'qa'         # 'qa' separates Passage/Question/Answer from User/Assistant chat; 'role' is legacy
DIALOGUE_WIDEN_L3 = False      # Optional: widen Layer 3 as well as Layer 7 to DIALOGUE_CONTEXT (0 new params)
LOCAL_DIALOGUE_DIR = ''        # Optional offline directory with both CoQA files and self-dialogs.json
SAMPLE_BYTES = 360             # Room for several sentences from the fixed river prompt
PROSE_COMPARISON_PARAGRAPHS = 3  # Validation paragraphs for paired free generation (0 skips)
SAMPLE_TEMPERATURE = .75
SAMPLE_TOP_P = .90             # 1.0 samples the full distribution

assert DEVICE in ('cpu', 'cuda') and TRAIN_MINUTES >= 0 and LR > 0
assert CONTEXT >= 128 and CONTEXT % 64 == 0 and BATCH_BYTES >= max(2048, 2*CONTEXT)
assert not AUTO_BATCH or MAX_BATCH_BYTES >= BATCH_BYTES
assert 0 < GPU_MEMORY_FRACTION <= 1
assert VALID_WINDOWS >= 2 and TEST_WINDOWS >= 2 and PROBE_PAIRS > 0
assert TASK_MINUTES >= 0 and SAMPLE_BYTES >= 0 and SAMPLE_TEMPERATURE > 0 and 0 < SAMPLE_TOP_P <= 1
assert PROSE_COMPARISON_PARAGRAPHS >= 0
assert FACT_SWITCH_MINUTES >= 0 and FACT_SWITCH_LR > 0 and FACT_SWITCH_EVAL_PAIRS >= 2
assert SHOWDOWN_STEPS >= 10
assert DIALOGUE_MINUTES >= 0 and DIALOGUE_LR > 0 and DIALOGUE_EVAL_ROWS >= 1
assert DIALOGUE_CONTEXT >= 512 and DIALOGUE_CONTEXT % 64 == 0
assert QA_PROMPT_STYLE in ('qa', 'role')
assert MAX_UPDATES is None or MAX_UPDATES >= 1
assert not (INITIAL_WEIGHTS and RESUME_FROM)

## Verify the runtime and choose an output folder

The setup checks a BF16 matrix multiply. It tries the optional fused gated-delta kernel and can use the CUDA PyTorch implementation.

In [ ]:
import contextlib, gc, hashlib, importlib, importlib.metadata, importlib.util, json
import os, shutil, statistics, subprocess, sys, tempfile, time

def pip_install(*requirements, required=True):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                           '--retries', '1', *requirements], check=required).returncode

if importlib.util.find_spec('torch') is None:
    pip_install('torch==2.9.1+cu128', '--index-url', 'https://download.pytorch.org/whl/cu128')
import torch
if DEVICE == 'cuda':
    if shutil.which('nvidia-smi'):
        subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'])
    if not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()):
        raise RuntimeError('Select an A100 GPU runtime in Colab, reconnect, and run all cells.')
    x = torch.ones((16, 16), device=DEVICE, dtype=torch.bfloat16)
    assert (x @ x)[0, 0].item() == 16
    del x
for module, package in [('numpy', 'numpy'), ('pyarrow', 'pyarrow'), ('huggingface_hub', 'huggingface-hub'), ('matplotlib', 'matplotlib')]:
    if importlib.util.find_spec(module) is None:
        pip_install(package)
import numpy as np
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

backend = 'torch'
if DEVICE == 'cuda' and TRY_FUSED:
    try:
        from fla.ops.gated_delta_rule import chunk_gated_delta_rule
        backend = 'fla'
    except ImportError:
        # Keep the working CUDA Torch/Triton pair while resolving optional dependencies.
        with tempfile.TemporaryDirectory() as tmp:
            pins = Path(tmp)/'constraints.txt'
            versions = {d: importlib.metadata.version(d) for d in ('torch', 'triton')
                        if importlib.util.find_spec(d)}
            pins.write_text(''.join(f'{d}=={v}\n' for d, v in versions.items()))
            if pip_install('-c', str(pins), 'fla-core', required=False) == 0:
                importlib.invalidate_caches()
                try:
                    from fla.ops.gated_delta_rule import chunk_gated_delta_rule
                    backend = 'fla'
                except ImportError as exc:
                    print('Using PyTorch gated delta:', exc)

local_root = Path('/content') if Path('/content').is_dir() else Path.cwd()
parent = Path(OUTPUT_ROOT) if OUTPUT_ROOT else local_root/'HGRT-10M'/'runs'
if SAVE_TO_DRIVE and not OUTPUT_ROOT:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        mount = Path('/content/drive/MyDrive')
        if not mount.is_dir():
            raise OSError('MyDrive unavailable')
        parent = mount/'HGRT-10M'/'runs'
    except Exception as exc:
        print(f'Drive unavailable ({type(exc).__name__}). This run will be stored on the runtime disk.')
parent.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix=time.strftime('wiki-%Y%m%d-%H%M%S-'), dir=parent))
data_cache = tempfile.TemporaryDirectory(prefix='hgrt-wikitext-')
cache = Path(data_cache.name)  # Fast local disk for this session
device = torch.device(DEVICE)
torch.manual_seed(SEED)

def precision(dev):
    return torch.autocast('cuda', dtype=torch.bfloat16) if torch.device(dev).type == 'cuda' else contextlib.nullcontext()

def sync():
    if device.type == 'cuda':
        torch.cuda.synchronize(device)

def gpu_stats():
    if DEVICE != 'cuda':
        return {}
    stats = dict(peak_allocated_gib=round(torch.cuda.max_memory_allocated(device)/2**30, 2))
    try:
        stats['recent_gpu_busy_pct'] = torch.cuda.utilization(device)
    except Exception:
        pass  # Utilization telemetry is optional on hosted GPUs.
    return stats

def save_json(name, obj):
    path = run_dir/name
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False))
    tmp.replace(path)

def event(**row):
    with (run_dir/'metrics.jsonl').open('a') as f:
        print(json.dumps(row, allow_nan=False), file=f)
    print(row, flush=True)

hardware = dict(device=torch.cuda.get_device_name() if DEVICE == 'cuda' else 'CPU',
                capacity_bytes=torch.cuda.get_device_properties(device).total_memory if DEVICE == 'cuda' else None,
                precision='bf16 autocast' if DEVICE == 'cuda' else 'fp32', torch=str(torch.__version__),
                cuda_build=torch.version.cuda, numpy=np.__version__)
print(hardware, '\nResults:', run_dir)

## Model

This is the project’s eight-layer C1-S8 implementation with 10,013,977 trainable parameters, six gated delta layers, two local attention layers, 64-byte chunks and eight tree slots. Each UTF-8 byte maps directly to one of 256 IDs.

In [ ]:
#@title HGRT-10M model definition
"""Eight-layer causal LM. Training builds binary-tree levels in parallel."""
import math
from dataclasses import dataclass

import torch
from torch import nn
from torch.nn import functional as F


@dataclass(frozen=True)
class Shape:
    width: int = 256
    heads: int = 4
    chunk: int = 64
    slot_dim: int = 128
    vocab: int = 256
    target: int = 10_000_000


def linear(a, b):
    return nn.Linear(a, b, bias=False)


def delta_reference(q, k, v, g, beta, state):
    """Solve the gated delta recurrence in small blocks, with FP32 accumulation."""
    b, length, h, d = q.shape
    block = min(32, length)
    pad = (-length) % block

    def blocks(x):
        x = F.pad(x.float(), (0, 0, 0, 0, 0, pad))
        return x.transpose(1, 2).reshape(b, h, -1, block, x.size(-1))

    with torch.autocast(q.device.type, enabled=False):
        qb, kb, vb = map(blocks, (q, k, v))
        beta = blocks(beta.unsqueeze(-1))
        log_decay = blocks(g.unsqueeze(-1)).cumsum(-2)
        decay = (log_decay - log_decay.transpose(-1, -2)).clamp_max(0).exp().tril()
        prefix = log_decay.exp()
        bk = beta * kb
        matrix = ((bk @ kb.transpose(-1, -2)) * decay).tril(-1)
        matrix = matrix + torch.eye(block, device=q.device)
        rhs = torch.cat((beta * vb, bk * prefix), -1)
        u, w = torch.linalg.solve_triangular(matrix, rhs, upper=False, unitriangular=True).chunk(2, -1)
        scores = (qb @ kb.transpose(-1, -2)) * decay
        end_keys = kb * decay[..., -1, :].unsqueeze(-1)
        if state is None:
            state = q.new_zeros((b, h, d, d), dtype=torch.float32)
        results = []
        for i in range(kb.size(2)):
            updates = u[:, :, i] - w[:, :, i] @ state
            results.append((qb[:, :, i] * prefix[:, :, i]) @ state + scores[:, :, i] @ updates)
            state = prefix[:, :, i, -1:] * state + end_keys[:, :, i].transpose(-1, -2) @ updates
        out = torch.cat(results, 2).transpose(1, 2)[:, :length]
    return out.to(q.dtype), state


class Delta(nn.Module):
    def __init__(self, s, backend):
        super().__init__()
        self.h, self.d = s.heads, s.width // s.heads
        self.backend = backend
        self.qkv = linear(s.width, 3 * s.width)
        self.conv = nn.Conv1d(3*s.width, 3*s.width, 4, groups=3*s.width, bias=False)
        self.gates = linear(s.width, 2*s.heads)
        self.decay_bias = nn.Parameter(torch.linspace(-5, -2, s.heads))
        self.out_gate = linear(s.width, s.width)
        self.out = linear(s.width, s.width)

    def forward(self, x, cache, offset, streaming):
        qkv = self.qkv(x).transpose(1, 2)
        prev, state = cache if cache is not None else (None, None)
        history = F.pad(qkv, (3, 0)) if prev is None else torch.cat((prev, qkv), -1)
        shaped = F.silu(self.conv(history).transpose(1, 2)).reshape(*x.shape[:2], 3, self.h, self.d)
        q, k, v = shaped.unbind(2)
        q, k = (F.normalize(t.float(), dim=-1, eps=1e-6).to(v.dtype) for t in (q, k))
        decay, beta = self.gates(x).float().chunk(2, -1)
        g = -F.softplus(decay + self.decay_bias)
        beta = beta.sigmoid()
        if self.backend == 'fla':
            from fla.ops.gated_delta_rule import chunk_gated_delta_rule
            out, state = chunk_gated_delta_rule(q.contiguous(), k.contiguous(), v.contiguous(),
                g.contiguous(), beta.to(v.dtype).contiguous(), scale=1., initial_state=state,
                output_final_state=streaming)
        else:
            out, state = delta_reference(q, k, v, g, beta, state)
        out = F.rms_norm(out, (self.d,), eps=1e-5).reshape_as(x)
        return self.out(out * F.silu(self.out_gate(x))), ((history[..., -3:].clone(), state) if streaming else None)


class Attention(nn.Module):
    def __init__(self, s, local):
        super().__init__()
        self.h, self.d = s.heads, s.width // s.heads
        self.window = s.chunk if local else 0
        self.qkv = linear(s.width, 3*s.width)
        self.out = linear(s.width, s.width)
        self.register_buffer("freq", 10000 ** (-torch.arange(0, self.d, 2) / self.d), persistent=False)

    def forward(self, x, cache, offset, streaming):
        b, length, width = x.shape
        q, k, v = self.qkv(x).reshape(b, length, 3, self.h, self.d).permute(2, 0, 3, 1, 4)
        angle = torch.arange(offset, offset+length, device=x.device)[:, None] * self.freq
        co, si = angle.cos().to(q.dtype), angle.sin().to(q.dtype)

        def rotate(t):
            a, z = t.chunk(2, -1)
            return torch.cat((a*co - z*si, z*co + a*si), -1)

        q, k = rotate(q), rotate(k)
        if cache is not None:
            k, v = (torch.cat((a, z), 2) for a, z in zip(cache, (k, v)))
        if cache is None and (not self.window or length <= self.window):
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        elif self.window and not streaming:
            # Each block sees the previous and current block, with an exact sliding mask.
            w = self.window
            pad = (-length) % w
            def split(t):
                return F.pad(t, (0, 0, 0, pad)).reshape(b, self.h, -1, w, self.d).transpose(1, 2)
            def adjacent(t):
                cur = split(t)
                prev = F.pad(cur[:, :-1], (0, 0, 0, 0, 0, 0, 1, 0))
                return torch.cat((prev, cur), 3)
            i = torch.arange(w, device=x.device)[:, None] + w
            j = torch.arange(2*w, device=x.device)
            mask = ((i >= j) & (i-j < w)).expand((length+pad)//w, 1, w, 2*w).clone()
            mask[0, :, :, :w] = False
            # Batch the windows in 4D so CUDA can dispatch its fused SDPA kernels.
            query = split(q).reshape(-1, self.h, w, self.d)
            keys = adjacent(k).reshape(-1, self.h, 2*w, self.d)
            values = adjacent(v).reshape_as(keys)
            y = F.scaled_dot_product_attention(query, keys, values, attn_mask=mask.repeat(b, 1, 1, 1))
            y = y.reshape(b, -1, self.h, w, self.d).transpose(1, 2).reshape(b, self.h, -1, self.d)[:, :, :length]
        else:
            difference = torch.arange(length, device=x.device)[:, None] + k.size(2)-length - torch.arange(k.size(2), device=x.device)
            valid = difference >= 0
            if self.window:
                valid &= difference < self.window
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=valid)
        keep = self.window-1 if self.window else k.size(2)
        next_cache = (k[:, :, -keep:].clone(), v[:, :, -keep:].clone()) if streaming else None
        return self.out(y.transpose(1, 2).reshape(b, length, width)), next_cache


class Tree(nn.Module):
    def __init__(self, s, slots):
        super().__init__()
        self.chunk = s.chunk
        self.project = linear(s.width, s.slot_dim)
        self.slots = nn.Parameter(torch.randn(slots, s.slot_dim) * 0.2)
        self.merge_proj = linear(s.slot_dim, s.slot_dim)
        nn.init.eye_(self.merge_proj.weight)
        self.gain = nn.Parameter(torch.tensor(0.1))

    def write(self, source):
        z = self.project(source)
        weights = (self.slots.to(z.dtype) @ z.transpose(-1, -2)) / math.sqrt(z.size(-1))
        return weights.softmax(-1) @ z

    def merge(self, left, right):
        return self.merge_proj((left + right) * .5)

    def read(self, x, nodes, mask=None):
        q = self.project(x)
        a = F.scaled_dot_product_attention(q, nodes, nodes, attn_mask=mask)
        return self.gain * F.linear(a, self.project.weight.T)

    def forward(self, x, source, mode):
        if mode == 'off' or x.size(1) <= self.chunk:
            return x
        if mode == 'shuffle':
            if len(x) < 2:
                raise ValueError("Shuffled memory requires independent donor sequences")
            source = source.roll(1, 0)
        b, length, d = x.shape
        c = self.chunk
        padded = F.pad(x, (0, 0, 0, (-length) % c)).reshape(b, -1, c, d)
        chunks = padded.size(1)
        level = self.write(source[:, :(chunks-1)*c].reshape(b, chunks-1, c, d))
        completed = torch.arange(chunks, device=x.device)
        nodes, masks = [], []
        span = 1
        while True:
            index = completed // span
            nodes.append(level[:, (index-1).clamp(0, level.size(1)-1)])
            masks.append((index % 2 == 1)[:, None].expand(-1, level.size(2)))
            pairs = level.size(1) // 2
            if not pairs:
                break
            level = self.merge(level[:, :2*pairs:2], level[:, 1:2*pairs:2])
            span *= 2
        # Chunk zero has no history. B,C act as batch and head axes in SDPA.
        memory = torch.cat(nodes, 2)[:, 1:]
        mask = torch.cat(masks, 1)[1:, None, :]
        reads = self.read(padded[:, 1:], memory, mask)
        return x + F.pad(reads.flatten(1, 2), (0, 0, c, 0))[:, :length]


class Block(nn.Module):
    def __init__(self, s, mix, hidden):
        super().__init__()
        self.mix = mix
        self.n1 = nn.RMSNorm(s.width, eps=1e-5)
        self.n2 = nn.RMSNorm(s.width, eps=1e-5)
        self.up, self.down = linear(s.width, 2*hidden), linear(hidden, s.width)

    def forward(self, x, cache, offset, streaming):
        y, cache = self.mix(self.n1(x), cache, offset, streaming)
        x = x + y
        gate, val = self.up(self.n2(x)).chunk(2, -1)
        return x + self.down(F.silu(gate)*val), cache


class LM(nn.Module):
    def __init__(self, kind='C1', slots=8, s=Shape(), backend='torch', read_window=None, mid_window=None):
        super().__init__()
        if kind not in ('A0', 'B1', 'C1'):
            raise ValueError(kind)
        self.s, self.kind = s, kind
        self.embed = nn.Embedding(s.vocab, s.width)
        nn.init.normal_(self.embed.weight, std=.02)
        self.norm = nn.RMSNorm(s.width, eps=1e-5)
        mixers = [Attention(s, kind != 'A0') if kind == 'A0' or i in (3, 7)
                  else Delta(s, backend) for i in range(8)]
        if read_window is not None:
            if kind != 'C1' or read_window < s.chunk:
                raise ValueError('read_window requires C1 and at least one full chunk')
            mixers[-1].window = read_window
        if mid_window is not None:
            if kind != 'C1' or mid_window < s.chunk:
                raise ValueError('mid_window requires C1 and at least one full chunk')
            mixers[3].window = mid_window
        # Centre the shared B1/C1 width between the two parameter counts.
        fixed = sum(p.numel() for module in [self.embed, self.norm, *mixers] for p in module.parameters()) + 16*s.width
        tree_mid = (s.width*s.slot_dim + s.slot_dim**2 + 16*s.slot_dim + 1)/2 if kind != 'A0' else 0
        self.hidden = max(4, 4 * round((s.target-fixed-tree_mid)/(8*3*s.width*4)))
        self.blocks = nn.ModuleList(Block(s, m, self.hidden) for m in mixers)
        self.tree = Tree(s, slots) if kind == 'C1' else None

    def forward(self, tokens, mode='on', positions=None):
        x = self.embed(tokens)
        for i, layer in enumerate(self.blocks):
            if i == 7 and self.tree is not None:
                x = self.tree(x, source, mode)
            x, _ = layer(x, None, 0, False)
            if i == 3:
                source = x
        if positions is not None:
            if positions.ndim == 1:
                x = x.index_select(1, positions)
            else:
                x = x[torch.arange(x.size(0), device=x.device)[:, None], positions]
        return F.linear(self.norm(x), self.embed.weight)

    def stream_chunk(self, tokens, state=None):
        """Process one complete aligned chunk, or a terminal partial chunk."""
        caches, forest, offset = state or ([None]*8, {}, 0)
        if offset % self.s.chunk or not 0 < tokens.size(1) <= self.s.chunk:
            raise ValueError("stream_chunk expects aligned chunks")
        x = self.embed(tokens)
        new_caches = []
        for i, layer in enumerate(self.blocks):
            if i == 7 and self.tree is not None and forest:
                x = x + self.tree.read(x, torch.cat([forest[j] for j in sorted(forest)], 1))
            x, cache = layer(x, caches[i], offset, True)
            new_caches.append(cache)
            if i == 3:
                source = x
        forest = dict(forest)
        if self.tree is not None and tokens.size(1) == self.s.chunk:
            node = self.tree.write(source)
            level = 0
            while level in forest:
                node = self.tree.merge(forest.pop(level), node)
                level += 1
            forest[level] = node
        return F.linear(self.norm(x), self.embed.weight), (new_caches, forest, offset + tokens.size(1))

In [ ]:
import codecs, re

def display_wiki_sample(raw):
    text = raw.replace(' @,@ ', ',').replace(' @.@ ', '.').replace(' @-@ ', '-')
    return re.sub(r'(?<=\S) +([,.!?;:])(?=\s|$)', r'\1', text)

def utf8_mask(ids, pending, remaining):
    if pending:
        # The first continuation excludes overlong encodings, surrogates and values above U+10FFFF.
        first = pending[0] if len(pending) == 1 else None
        low = {0xE0: 0xA0, 0xF0: 0x90}.get(first, 0x80)
        high = {0xED: 0x9F, 0xF4: 0x8F}.get(first, 0xBF)
        return (ids >= low) & (ids <= high)
    return ((ids < 0x80) | ((remaining >= 2) & (ids >= 0xC2) & (ids <= 0xDF)) |
            ((remaining >= 3) & (ids >= 0xE0) & (ids <= 0xEF)) |
            ((remaining >= 4) & (ids >= 0xF0) & (ids <= 0xF4)))

def sample_byte(logits, temperature, top_p, generator):
    probs = (logits.float() / temperature).softmax(-1)
    ordered, ids = probs.sort(descending=True)
    # Keep the first byte that crosses the threshold, even if it is the only one.
    keep = ordered.cumsum(0) - ordered < top_p
    index = torch.multinomial(ordered.masked_fill(~keep, 0), 1, generator=generator)
    return int(ids[index].item())

def finished_sentence(tail):
    return re.search(rb'[.!?]["\x27)]?\s$', tail) is not None

@torch.inference_mode()
def sample(model, prompt, count, temperature, device, top_p=1., seed=123, finish_after=None, stop_at_newline=False):
    tokens = list(prompt.encode('utf8'))
    if not tokens:
        raise ValueError('Supply a nonempty prompt')
    generator = torch.Generator(device=device).manual_seed(seed)
    decoder = codecs.getincrementaldecoder('utf8')()
    byte_ids = torch.arange(256, device=device)
    chunk = model.s.chunk
    state = None
    # A full chunk can be committed once. Only recompute the unfinished suffix.
    start = 0
    for remaining in range(count, 0, -1):
        while len(tokens)-start > chunk:
            ids = torch.tensor([tokens[start:start+chunk]], device=device)
            _, state = model.stream_chunk(ids,state)
            start += chunk
        ids = torch.tensor([tokens[start:]], device=device)
        logits, pending = model.stream_chunk(ids,state)
        valid = utf8_mask(byte_ids, decoder.getstate()[0], remaining)
        byte = sample_byte(logits[0,-1].float().masked_fill(~valid, -torch.inf), temperature, top_p, generator)
        decoder.decode(bytes([byte]))
        tokens.append(byte)
        if stop_at_newline and byte == 10:
            break
        generated = count - remaining + 1
        if finish_after is not None and generated >= finish_after and byte in b' \n\t\r':
            if finished_sentence(bytes(tokens[-min(generated, 3):])):
                break
        if ids.size(1) == chunk:
            state = pending
            start += chunk
    return bytes(tokens).decode('utf8')

## Prepare the corpus

Read the two train shards and the official validation and test shards separately. Each row contributes its UTF-8 text with a terminating newline. Blank rows become a newline. Local memory-mapped byte files keep the full corpus out of GPU memory. Ordinary training windows are drawn with replacement from the train split. For continuation practice, the loader also indexes sentence transitions with sufficient text on both sides inside one paragraph. Each eligible paragraph contributes one boundary. Fixed nonoverlapping validation and test windows are selected independently. Split hashes, byte counts and exact offsets are saved. A local Parquet override is labeled with its actual path and hashes. [Pinned upstream revision](https://huggingface.co/datasets/Salesforce/wikitext/commit/b08601e04326c79dfdd32d625aee71d232d685c3)

In [ ]:
import re
from array import array

REPO = 'Salesforce/wikitext'
REV = 'b08601e04326c79dfdd32d625aee71d232d685c3'
SUBSET = 'wikitext-103-raw-v1'
FILES = {'train': ['train-00000-of-00002.parquet', 'train-00001-of-00002.parquet'],
         'validation': ['validation-00000-of-00001.parquet'], 'test': ['test-00000-of-00001.parquet']}

prefix_bytes, ending_bytes = min(192, CONTEXT//2), min(128, CONTEXT//2)

def sentence_cuts(paragraph):
    for match in re.finditer(rb"""[.!?]["']?[ \t]+(?=[A-Z"'])""", paragraph):
        if paragraph[match.start()] == ord('.'):
            word = paragraph[max(0, match.start()-12):match.start()].rstrip()
            if re.search(rb'\b(?:Mr|Mrs|Ms|Dr|Prof|St|Ste|Mt|Gen|Sr|Jr|[A-Z])$', word):
                continue
        yield match.end()

def sentence_boundary(paragraph, left, right):
    return next((cut for cut in sentence_cuts(paragraph) if left <= cut <= len(paragraph)-right), None)

def make_split(split):
    path = cache/f'{split}.bin'
    digest = hashlib.sha256()
    count, offset = 0, 0
    anchors = array('q')
    with path.open('wb') as dst:
        for name in FILES[split]:
            shard = (Path(LOCAL_PARQUET_DIR)/name if LOCAL_PARQUET_DIR else
                     hf_hub_download(REPO, f'{SUBSET}/{name}', repo_type='dataset', revision=REV))
            for batch in pq.ParquetFile(shard).iter_batches(columns=['text'], batch_size=4096):
                lines = batch.column(0).to_pylist()
                raw = bytearray()
                for line in lines:
                    row = (line if line.endswith('\n') else line+'\n').encode('utf8')
                    for paragraph in row.splitlines(keepends=True):
                        cut = sentence_boundary(paragraph.rstrip(b'\r\n'), prefix_bytes, ending_bytes)
                        if cut is not None:
                            anchors.append(offset + cut)
                        offset += len(paragraph)
                    raw.extend(row)
                dst.write(raw)
                digest.update(raw)
                count += len(lines)
    if path.stat().st_size < max(2*CONTEXT+1, 2049):
        raise ValueError(f'{split} split too small for the selected benchmark')
    return (np.memmap(path, dtype=np.uint8, mode='c'),
            dict(bytes=path.stat().st_size, rows=count, sha256=digest.hexdigest(), files=FILES[split],
                 continuation_paragraphs=len(anchors)), np.asarray(anchors, dtype=np.int64))

corpus, split_info, anchors = {}, {}, {}
for split in FILES:
    corpus[split], split_info[split], anchors[split] = make_split(split)
    print(split, split_info[split]['rows'], 'rows,', len(corpus[split]), 'bytes')
# On Windows the byte mappings must be closed before the temporary directory is removed.
import atexit
def close_corpus(maps, tmp_dir=None):
    for raw in maps.values():
        try:
            raw._mmap.close()
        except Exception:
            pass
    if tmp_dir is not None:
        try:
            tmp_dir.cleanup()
        except Exception:
            pass
atexit.register(close_corpus, corpus, data_cache)

data_manifest = dict(source=str(LOCAL_PARQUET_DIR) if LOCAL_PARQUET_DIR else REPO,
                     revision=None if LOCAL_PARQUET_DIR else REV, subset=SUBSET, splits=split_info)
save_json('corpus.json', data_manifest)

from functools import lru_cache

train_storage = corpus['train']
if DEVICE == 'cuda' and train_storage.nbytes <= .10 * torch.cuda.mem_get_info(device)[0] * GPU_MEMORY_FRACTION:
    try:
        train_storage = torch.from_numpy(train_storage).to(device)
    except torch.OutOfMemoryError:
        pass  # The memory-mapped training split is already available.
print('Training windows:', 'GPU byte cache' if isinstance(train_storage, torch.Tensor) else 'local memory map')

@lru_cache(maxsize=4)
def window_offsets(length, dev):
    return torch.arange(length + 1, device=dev)

def windows(raw, starts, length):
    if isinstance(raw, torch.Tensor):
        offsets = torch.as_tensor(starts).to(raw.device, non_blocking=True)
        ids = raw[offsets[:, None] + window_offsets(length, raw.device)].long()
    else:
        block = raw[np.asarray(starts)[:, None] + np.arange(length+1)]
        ids = torch.from_numpy(block).to(device, non_blocking=True).long()
    return ids[:, :-1], ids[:, 1:]

def train_batch(rng, length, batch):
    raw = train_storage
    return windows(raw, rng.integers(0, len(raw)-length, size=batch), length)

def choose_paragraphs(rng, count, size):
    real = rng.integers(count, size=size)
    other = rng.integers(count-1, size=size)
    other += (other >= real)
    # Nearby paragraphs often share vocabulary and entities, making a harder foil.
    nearby = size // 2
    if nearby:
        local = real[:nearby]
        lo, hi = np.maximum(0, local-8), np.minimum(count-1, local+8)
        alt = rng.integers(lo, hi)
        other[:nearby] = alt + (alt >= local)
    return real, other

def continuation_batch(raw, cuts, real, other, left, right):
    x, y = windows(raw, cuts[real] - left, left+right-1)
    other_prefix, _ = windows(raw, cuts[other] - left, left)
    foil = torch.cat((other_prefix, x[:, left:]), dim=1)
    return torch.cat((x, foil)), y

def continuation_nll(logits, targets, left):
    good, foil = logits.float().chunk(2, dim=0)
    matched = F.cross_entropy(good.transpose(1, 2), targets, reduction='none')
    mismatched = F.cross_entropy(foil[:, left-1:].transpose(1, 2), targets[:, left-1:], reduction='none')
    return matched, mismatched

use_continuations = len(anchors['train']) >= 2 and len(anchors['validation']) >= 2
if not use_continuations:
    print('Too few eligible paragraphs for continuation practice. Using ordinary next-byte training.')

def fixed_starts(raw, n, seed):
    # Disjoint target bytes; randomize batch neighbours for the shuffled-tree control.
    eligible = (len(raw)-1)//CONTEXT
    n = min(n, eligible)//2*2
    if n < 2:
        raise ValueError('Need two non-overlapping evaluation windows')
    return np.random.default_rng(seed).choice(eligible, n, replace=False)*CONTEXT

starts = {s: fixed_starts(corpus[s], n, k) for s,n,k in
          [('validation', VALID_WINDOWS, 3100), ('test', TEST_WINDOWS, 9413)]}
save_json('eval_windows.json', {s:p.tolist() for s,p in starts.items()})

def fit_byte_baselines(raw):
    uni = np.ones(256, dtype=np.int64)
    bi = np.ones(65536, dtype=np.int64)
    for i in range(0, len(raw), 1<<20):
        piece = raw[i:i+(1<<20)]
        uni += np.bincount(piece, minlength=256)
        size = min(len(piece), len(raw)-1-i)
        if size:
            transitions = piece[:size].astype(np.int32)*256 + raw[i+1:i+1+size]
            bi += np.bincount(transitions, minlength=65536)
    bi = bi.reshape(256,256)
    return -np.log2(uni/uni.sum()), -np.log2(bi/bi.sum(1, keepdims=True))

uni_bits, bi_bits = fit_byte_baselines(corpus['train'])

def baseline_scores(split):
    blocks = corpus[split][starts[split][:, None]+np.arange(CONTEXT+1)]
    x, y = blocks[:,:-1], blocks[:,1:]
    return dict(unigram_bpb=float(uni_bits[y].mean()), bigram_bpb=float(bi_bits[x,y].mean()))

## Benchmark the selected hardware

The notebook checks the tree gradients at 2,048 bytes, then times real training updates at the requested context. On CUDA it increases the batch while measuring bytes per second, with a configurable memory reserve. Practice runs its own search on full questions and measures pairs per second. Each candidate gets one warm-up and three timed updates on disposable weights. Timings include batch preparation, backward and AdamW. The measured winner, candidate rates and peak allocations are saved. CPU runs use the starting budget. To fix the starting batch on GPU, set `AUTO_BATCH = False`.

In [ ]:
def new_model(kernel):
    torch.manual_seed(SEED)
    model = LM('C1', 8, backend=kernel).to(device)
    assert sum(p.numel() for p in model.parameters()) == 10_013_977
    return model

def make_optimizer(model):
    return torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=.01, fused=(DEVICE == 'cuda'))

def prose_backward(model, optimizer, x, y, check_tree=False, continuation=False):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    with precision(device):
        logits = model(x)
        if continuation:
            matched, mismatched = continuation_nll(logits, y, prefix_bytes)
            margin = (mismatched - matched[:, prefix_bytes-1:]).sum(1) / math.sqrt(ending_bytes)
            ending = matched[:, prefix_bytes-1:]
            # Give the start of the next sentence its own prediction signal.
            loss = .25*matched.mean() + .50*ending.mean() + .25*ending[:, :32].mean()
            loss = loss + .25*F.softplus(-margin).mean()
        else:
            loss = F.cross_entropy(logits.reshape(-1,256).float(), y.reshape(-1))
    loss.backward()
    norms = {}
    if check_tree:
        norms = {name: float(p.grad.norm()) if p.grad is not None else 0.
                 for name,p in model.tree.named_parameters()}
        if not all(math.isfinite(v) and v > 0 for v in norms.values()):
            raise FloatingPointError(f'Missing finite tree gradients: {norms}')
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)
    return loss.detach(), norms

def release_memory():
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

def allocation_limit():
    if DEVICE != 'cuda' or not AUTO_BATCH:
        return None
    free, _ = torch.cuda.mem_get_info(device)
    # Reserved blocks already belong to this process and can be reused.
    return int((free + torch.cuda.memory_reserved(device)) * GPU_MEMORY_FRACTION)

def timed_updates(update_once):
    if DEVICE == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
    update_once()
    sync()
    times = []
    for _ in range(3):
        start = time.perf_counter()
        update_once()
        sync()
        times.append(time.perf_counter() - start)
    peak = torch.cuda.max_memory_allocated(device) if DEVICE == 'cuda' else None
    return statistics.median(times), peak

def choose_batch(measure, initial, minimum, maximum, memory_limit, rate_key):
    size = min(initial, maximum)
    best, trials, plateau = None, [], 0
    last_fit = None
    while True:
        try:
            trial = measure(size)
        except torch.OutOfMemoryError:
            if size == minimum and best is None:
                raise
            trial = dict(batch=size, status='out_of_memory')
        peak = trial.get('peak_allocated_bytes')
        if memory_limit is not None and peak is not None and peak > memory_limit:
            trial['status'] = 'over_memory_budget'
        trial.setdefault('status', 'ok')
        trials.append(trial)
        print(f"  batch {size}: {trial['status']}", end='')
        if trial['status'] != 'ok':
            print()
            release_memory()
            if last_fit is not None:
                previous = last_fit['batch']
                if size - previous <= max(1, previous//8):
                    break
                maximum = size - 1
                size = (size + previous)//2
                continue
            if size == minimum:
                raise MemoryError('Even the minimum batch exceeds GPU_MEMORY_FRACTION. Check available GPU memory.')
            maximum = size - 1
            size = max(minimum, size // 2)
            continue
        rate = trial[rate_key]
        print(f' ({rate:,.0f} {rate_key})')
        plateau = plateau + 1 if best is not None and rate < best[rate_key]*1.03 else 0
        if best is None or rate > best[rate_key]:
            best = trial
        if size == maximum or plateau >= 2:
            break
        next_size = min(maximum, 2*size)
        if memory_limit is not None and last_fit is not None and peak > last_fit['peak_allocated_bytes']:
            bytes_per_example = (peak-last_fit['peak_allocated_bytes'])/(size-last_fit['batch'])
            room = int((memory_limit-peak)/bytes_per_example)
            if room < max(1, size//16):
                break
            next_size = min(next_size, size+room)
        last_fit = trial
        size = next_size
    baseline = trials[0]
    if baseline['status'] == 'ok':
        print(f"  selected batch {best['batch']}, {best[rate_key]/baseline[rate_key]:.2f}x starting throughput")
    return best, trials

def benchmark(length, batch, kernel):
    model = new_model(kernel)
    opt = make_optimizer(model)
    rng = np.random.default_rng(800)
    norms = {}
    def one_update():
        x, y = train_batch(rng, length, batch)
        _, checked = prose_backward(model, opt, x, y, check_tree=(length == 2048 and not norms))
        norms.update(checked)
        opt.step()
    seconds, peak = timed_updates(one_update)
    x, _ = train_batch(rng, length, 1)
    with torch.inference_mode(), precision(device):
        model.eval()
        model(x)
        sync()
        latency = []
        for _ in range(5):
            start = time.perf_counter()
            model(x)
            sync()
            latency.append(1000*(time.perf_counter()-start))
    return dict(length=length, batch=batch, backend=kernel, timed_train_steps=3,
                median_train_step_seconds=seconds, train_bytes_per_second=batch*length/seconds,
                peak_allocated_bytes=peak, batch1_prefill_ms=statistics.median(latency),
                tree_gradient_norms=norms)

dynamic = DEVICE == 'cuda' and AUTO_BATCH
def profile_prose(kernel):
    limit = allocation_limit()
    diagnostic_batch = max(1, BATCH_BYTES//2048)
    diagnostic, _ = choose_batch(lambda b: benchmark(2048, b, kernel), diagnostic_batch,
                                 1, diagnostic_batch, limit, 'train_bytes_per_second')
    initial = BATCH_BYTES//CONTEXT
    maximum = MAX_BATCH_BYTES//CONTEXT if dynamic else initial
    def measure(batch):
        if CONTEXT == 2048 and batch == diagnostic['batch']:
            return dict(diagnostic)
        return benchmark(CONTEXT, batch, kernel)
    selected, trials = choose_batch(measure, initial, 2, maximum, limit, 'train_bytes_per_second')
    profiles = [diagnostic, selected] if CONTEXT != 2048 else [selected]
    return profiles, dict(selected_batch=selected['batch'], memory_limit_bytes=limit, trials=trials)

def setup_profiles(kernel):
    if TRAIN_MINUTES > 0:
        return profile_prose(kernel)
    if kernel == 'fla':
        # Check the optional kernel at practice length before choosing its fallback.
        check = new_model(kernel)
        tokens = torch.zeros(2, 128, device=device, dtype=torch.long)
        with precision(device):
            probe_loss = check(tokens)[:, -1].float().mean()
        probe_loss.backward()
        sync()
    print('Using saved prose weights. Skipping prose training benchmarks.')
    return [], dict(selected_batch=2, trials=[], skipped='practice_only')

try:
    bench, prose_tuning = setup_profiles(backend)
except Exception as exc:
    if backend != 'fla' or isinstance(exc, (MemoryError, torch.OutOfMemoryError, FloatingPointError, AssertionError)):
        raise
    print(f'FLA benchmark failed ({type(exc).__name__}: {exc}). Retrying with CUDA PyTorch.')
    backend = 'torch'
    bench = None
if bench is None:
    release_memory()
    bench, prose_tuning = setup_profiles(backend)
batch_size = prose_tuning['selected_batch']
# Keep validation and shuffled-tree donor groups comparable to a fixed-budget run.
eval_batch = max(2, min(BATCH_BYTES//CONTEXT, batch_size)//2*2)
save_json('benchmark.json', dict(hardware=hardware, profiles=bench, prose_tuning=prose_tuning))
if bench:
    print(f"Selected {batch_size} prose sequences per update, {bench[-1]['train_bytes_per_second']:,.0f} bytes/s.")

## Train the language model

AdamW uses BF16 autocast on GPU, gradient clipping at 1.0 and a linear 50-update learning-rate warm-up. Three out of four updates use ordinary next-byte training. On the fourth, each real sentence continuation is also scored after a different training paragraph. Half the foils come from nearby paragraphs. The prediction loss emphasizes the true continuation and its first 32 bytes. The loss rewards both predicting the real text and preferring its original context. Held-out comparisons use a fixed set of paragraphs, with the same target bytes in each comparison. Every 90 seconds of loop time the notebook checks validation and saves `latest.pt`. It also saves at the final update. The lowest held-out continuation BPB observed in this invocation selects the prose weights (ordinary validation BPB on corpora with too few eligible paragraphs) for sampling, test evaluation and the initial reasoning probe. These weights are saved as `best_lm.pt`. Resume adds the configured minutes in a new output folder; data hashes, model, optimizer settings and seed must match. Automatic CUDA batching can resume with a different batch size. A smaller batch is retried if GPU memory becomes scarce during a forward or backward pass. An inherited checkpoint loaded through `INITIAL_WEIGHTS` starts a fresh optimizer.

In [ ]:
@torch.inference_mode()
def evaluate(model, split, mode='on'):
    model.eval()
    nll = torch.zeros((), dtype=torch.float64, device=device)
    total = 0
    for i in range(0, len(starts[split]), eval_batch):
        x, y = windows(corpus[split], starts[split][i:i+eval_batch], CONTEXT)
        with precision(device):
            logits = model(x, mode)
            nll += F.cross_entropy(logits.float().reshape(-1,256), y.reshape(-1), reduction='sum')
        total += y.numel()
    return float(nll)/(total*math.log(2))

@torch.inference_mode()
def evaluate_continuations(model, split):
    limit = VALID_WINDOWS if split == 'validation' else TEST_WINDOWS
    size = min(limit, len(anchors[split]))
    if size < 2:
        return None
    real = np.random.default_rng(3701 if split == 'validation' else 3702).choice(len(anchors[split]), size, replace=False)
    other = np.roll(real, 1)
    totals = torch.zeros(3, device=device, dtype=torch.float64)
    model.eval()
    for i in range(0, size, max(1, eval_batch//2)):
        sl = slice(i, i+max(1, eval_batch//2))
        ids, y = continuation_batch(corpus[split], anchors[split], real[sl], other[sl], prefix_bytes, ending_bytes)
        with precision(device):
            logits = model(ids)
        good, bad = continuation_nll(logits, y, prefix_bytes)
        g, b = good[:, prefix_bytes-1:].mean(1), bad.mean(1)
        totals += torch.stack((g.sum(), b.sum(), ((g < b).float() + .5*(g == b)).sum()))
    good, bad, wins = (totals / size).tolist()
    return dict(paragraphs=size, prefix_bytes=prefix_bytes, continuation_bytes=ending_bytes,
                matched_bpb=good/math.log(2), mismatched_bpb=bad/math.log(2), matching_accuracy=wins)

config = dict(kind='C1', slots=8, seed=SEED, context=CONTEXT, batch=batch_size, lr=LR,
              tokenizer='utf8-bytes', corpus_hashes={k:v['sha256'] for k,v in split_info.items()})

def compatible_config(saved, current, adaptive):
    if not isinstance(saved, dict):
        return False
    if adaptive and 'batch' in saved and 'batch' in current:
        return dict(saved, batch=current['batch']) == current
    return saved == current

def matching_checkpoint(roots, config):
    matches = []
    for root in roots:
        for path in root.glob('wiki-*/latest.pt'):
            if path.parent == run_dir:
                continue
            try:
                previous = json.loads((path.parent/'config.json').read_text(encoding='utf8'))
                if compatible_config(previous.get('config'), config, dynamic or TRAIN_MINUTES == 0):
                    matches.append((path.stat().st_mtime, path))
            except (OSError, ValueError):
                continue
    return max(matches)[1] if matches else None

resume_path = Path(RESUME_FROM) if RESUME_FROM else None
if AUTO_RESUME and not (resume_path or INITIAL_WEIGHTS):
    roots = {parent} if OUTPUT_ROOT else {parent, local_root/'HGRT-10M'/'runs'}
    resume_path = matching_checkpoint(roots, config)
    if resume_path is None:
        warm_roots = roots if OUTPUT_ROOT else roots | {Path.cwd()/'runs'}
        candidates = {path for root in warm_roots for path in
                      [root/'best_lm.pt', *root.glob('wiki-*/best_lm.pt')] if path.is_file()}
        for path in sorted(candidates, key=lambda p: (p.stat().st_mtime, str(p)), reverse=True):
            weights = torch.load(path, map_location='cpu', weights_only=True)
            matches = (weights.get('kind') == 'C1' and weights.get('slots') == 8 and
                       weights.get('config', {}).get('corpus_hashes') == config['corpus_hashes'])
            del weights
            if matches:
                INITIAL_WEIGHTS = str(path)
                break
if TRAIN_MINUTES == 0 and not (resume_path or INITIAL_WEIGHTS):
    raise ValueError('For practice without prose training, supply an existing prose checkpoint.')
print('Continuing from ' + str(resume_path) if resume_path else
      'Starting from ' + str(INITIAL_WEIGHTS) if INITIAL_WEIGHTS else 'Starting a fresh prose model.')
save_json('config.json', dict(config=config, hardware=hardware, backend=backend, minutes=TRAIN_MINUTES,
                              initial_weights=str(INITIAL_WEIGHTS), resume_from=str(resume_path or ''),
                              prose_objective='paragraph-continuation-v2', continuation_active=use_continuations))
model = new_model(backend)
optimizer = make_optimizer(model) if TRAIN_MINUTES else None
train_rng = np.random.default_rng(SEED)
step, bytes_seen = 0, 0
if resume_path or INITIAL_WEIGHTS:
    ck = torch.load(resume_path or INITIAL_WEIGHTS, map_location='cpu', weights_only=True)
    if ck['kind'] != 'C1' or ck['slots'] != 8:
        raise ValueError('Select a C1-S8 checkpoint.')
    model.load_state_dict(ck['state_dict'])
    model.blocks[-1].mix.window = ck.get('read_window') or model.s.chunk
    model.blocks[3].mix.window = ck.get('mid_window') or model.s.chunk
    if resume_path:
        if not compatible_config(ck.get('config'), config, dynamic or TRAIN_MINUTES == 0):
            raise ValueError('Resume requires matching data, model and optimizer settings. Use INITIAL_WEIGHTS for transfer.')
        previous_batch = ck.get('batch_at_save', ck['config']['batch'])
        if TRAIN_MINUTES and previous_batch != batch_size:
            print(f'Resuming with {batch_size} sequences per update (previously {previous_batch}).')
        if TRAIN_MINUTES:
            optimizer.load_state_dict(ck['optimizer'])
        step, bytes_seen = ck['step'], ck['bytes_seen']
        train_rng.bit_generator.state = json.loads(ck['train_rng'])
    else:
        step, bytes_seen = ck.get('step', 0), ck.get('bytes_seen', 0)
    del ck

def save_checkpoint():
    tmp = run_dir/'latest.partial'
    torch.save(dict(kind='C1', slots=8, read_window=model.blocks[-1].mix.window, mid_window=model.blocks[3].mix.window, config=config, state_dict=model.state_dict(),
                    optimizer=optimizer.state_dict(), step=step, bytes_seen=bytes_seen, batch_at_save=batch_size,
                    train_rng=json.dumps(train_rng.bit_generator.state)), tmp)
    tmp.replace(run_dir/'latest.pt')

best_bpb, best_step, best_selection_loss = float('inf'), None, float('inf')

def keep_best_lm(bpb):
    global best_bpb, best_step, best_selection_loss
    continuation = evaluate_continuations(model, 'validation') if use_continuations else None
    selection_loss = continuation['matched_bpb'] if continuation else bpb
    if not math.isfinite(bpb) or not math.isfinite(selection_loss):
        raise FloatingPointError(f'Nonfinite validation BPB at step {step}')
    if selection_loss < best_selection_loss:
        tmp = run_dir/'best_lm.partial'
        torch.save(dict(kind='C1', slots=8, read_window=model.blocks[-1].mix.window, mid_window=model.blocks[3].mix.window, config=config, state_dict=model.state_dict(),
                        step=step, bytes_seen=bytes_seen, validation_bpb=bpb, continuation=continuation), tmp)
        tmp.replace(run_dir/'best_lm.pt')
        best_bpb, best_step, best_selection_loss = bpb, step, selection_loss
    return continuation

before_bpb = evaluate(model, 'validation')
valid_base = baseline_scores('validation')
curve = [(step, before_bpb)]
initial_continuation = keep_best_lm(before_bpb)
event(stage='initial', step=step, bytes_seen=bytes_seen, validation_bpb=before_bpb, continuation=initial_continuation, **valid_base)

prompt = 'The expedition reached the river at dawn. '
before_passage = None
if TRAIN_MINUTES > 0 and (resume_path or INITIAL_WEIGHTS):
    with precision(device):
        before_passage = sample(model, prompt, SAMPLE_BYTES, SAMPLE_TEMPERATURE, device, SAMPLE_TOP_P,
                                finish_after=max(1, SAMPLE_BYTES//2))
    (run_dir/'sample_before_raw.txt').write_text(before_passage, encoding='utf8')

In [ ]:
def backward_with_backoff(backward, optimizer, rng, size, minimum, adaptive):
    # Retrying is safe only before AdamW starts mutating weights and moments.
    if not adaptive:
        return backward(size), size
    sampler_state = rng.bit_generator.state
    while True:
        try:
            return backward(size), size
        except torch.OutOfMemoryError:
            if size <= minimum:
                raise
        optimizer.zero_grad(set_to_none=True)
        release_memory()
        rng.bit_generator.state = sampler_state
        size = max(minimum, size//2)
        print(f'GPU memory pressure. Retrying the update with batch {size}.')

def train_prose_batch(size):
    contrast = use_continuations and (step+1) % 4 == 0
    if contrast:
        real, other = choose_paragraphs(train_rng, len(anchors['train']), max(1, size//2))
        x, y = continuation_batch(train_storage, anchors['train'], real, other, prefix_bytes, ending_bytes)
    else:
        x, y = train_batch(train_rng, CONTEXT, size)
    loss, _ = prose_backward(model, optimizer, x, y, continuation=contrast)
    return loss, y.numel()

initial_step, initial_bytes = step, bytes_seen
interval_bytes = bytes_seen
if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats(device)
start_clock = last_check = time.perf_counter()
deadline = start_clock + 60*TRAIN_MINUTES
while time.perf_counter() < deadline and (MAX_UPDATES is None or step-initial_step < MAX_UPDATES):
    warmup_step = step+1 if resume_path else step-initial_step+1
    for group in optimizer.param_groups:
        group['lr'] = LR*min(1., warmup_step/50)
    (loss, target_bytes), batch_size = backward_with_backoff(train_prose_batch, optimizer, train_rng, batch_size, 2, dynamic)
    optimizer.step()
    step += 1
    bytes_seen += target_bytes
    if time.perf_counter()-last_check >= 90:
        sync()
        rate = (bytes_seen-interval_bytes)/(time.perf_counter()-last_check)
        stats = gpu_stats()
        bpb = evaluate(model, 'validation')
        curve.append((step,bpb))
        save_checkpoint()
        cont = keep_best_lm(bpb)
        event(stage='train', step=step, bytes_seen=bytes_seen, loss=float(loss), validation_bpb=bpb, continuation=cont,
              batch=batch_size, training_bytes_per_second=rate, **stats)
        interval_bytes = bytes_seen
        last_check = time.perf_counter()
sync()
train_seconds = time.perf_counter()-start_clock
if step > initial_step:
    save_checkpoint()
resume_checkpoint = (run_dir/'latest.pt') if step > initial_step else (resume_path or INITIAL_WEIGHTS or None)
event(stage='train_done', step=step, bytes_seen=bytes_seen, additional_steps=step-initial_step,
      train_loop_seconds=train_seconds, batch=batch_size, bytes_per_second=(bytes_seen-initial_bytes)/train_seconds,
      checkpoint=str(resume_checkpoint) if resume_checkpoint else None)

## Evaluate held-out text

Bits per byte (BPB) is the average negative log₂ probability of the observed next byte; lower is better. Byte perplexity equals 2ᴮᴾᴮ. Compare models using the same byte targets and context. Train-fitted unigram and bigram baselines score exactly the same test windows. A positive disabled-minus-intact or shuffled-minus-intact difference indicates a benefit for the intact tree on these windows. The shuffled condition rotates completed tree histories across different batch windows. The reserved test subset is intended for the final report; use validation for model choices.

In [ ]:
validation_bpb = evaluate(model,'validation')
curve.append((step, validation_bpb))
keep_best_lm(validation_bpb)
from copy import deepcopy
prose_model = deepcopy(model).eval()
selected = torch.load(run_dir/'best_lm.pt', map_location='cpu', weights_only=True)
prose_model.load_state_dict(selected['state_dict'])
selected_continuation = selected.get('continuation')
selected_prose_bytes = selected['bytes_seen']
del selected
print(f'Using prose checkpoint from step {best_step} ({best_bpb:.3f} validation BPB).')
test_scores = {mode:evaluate(prose_model, 'test', mode) for mode in ('on','off','shuffle')}
test_base = baseline_scores('test')
lm_results = dict(initial_validation_bpb=before_bpb, final_validation_bpb=validation_bpb,
                  initial_continuation_validation=initial_continuation, selected_continuation_validation=selected_continuation,
                  selected_continuation_test=evaluate_continuations(prose_model, 'test'),
                  selected_validation_bpb=best_bpb, selected_step=best_step, selected_checkpoint=str(run_dir/'best_lm.pt'),
                  validation_targets=len(starts['validation'])*CONTEXT,
                  test_targets=len(starts['test'])*CONTEXT, test_bpb=test_scores,
                  intact_test_byte_perplexity=2**test_scores['on'], test_baselines=test_base,
                  off_minus_intact_bpb=test_scores['off']-test_scores['on'],
                  shuffle_minus_intact_bpb=test_scores['shuffle']-test_scores['on'],
                  optimizer_steps=step, training_bytes=bytes_seen, additional_steps=step-initial_step,
                  train_loop_seconds=train_seconds, final_batch=batch_size,
                  loop_bytes_per_second=(bytes_seen-initial_bytes)/train_seconds)
save_json('lm_results.json', lm_results)
print(json.dumps(lm_results, indent=2))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 3.6))
steps, bpb = zip(*curve)
ax.plot(steps, bpb, color='#18558a', marker='o', markersize=4, label='C1-S8')
ax.axhline(valid_base['unigram_bpb'], color='#666666', linestyle=':', label='Train unigram')
ax.axhline(valid_base['bigram_bpb'], color='#ac5400', linestyle='--', label='Train bigram')
ax.set(xlabel='Optimizer update', ylabel='Held-out bits per byte', title='WikiText validation windows' if not LOCAL_PARQUET_DIR else 'Validation windows')
ax.grid(axis='y', alpha=.2)
ax.legend()
fig.tight_layout()
fig.savefig(run_dir/'validation_curve.png', dpi=150)
plt.show()
plt.close(fig)

## Generate a sample

The sample uses the validation-selected prose checkpoint, the original river prompt and a fixed seed. When continuing learned weights, the notebook generates a passage before and after prose training with the same settings. Read both to judge whether later sentences follow the subject and events of the opening. The sampler uses valid UTF-8 nucleus sampling with `SAMPLE_TOP_P`. After half the byte budget, it stops at the next generated sentence-ending punctuation followed by whitespace, or at the hard byte limit. Both displays use the same punctuation cleanup. `sample_raw.txt` preserves the generated text.

The notebook also generates from several validation paragraphs twice at the same sentence boundary, once with just the last sentence and once with up to 192 bytes of preceding sentences. Each pair uses the same seed and gets no bytes from the reference continuation. Read across both seeds for whether the next sentences keep the same subject and events. Raw prompts and completions are saved in `prose_comparison.json`.

In [ ]:
def prose_prompt_pairs(raw, cuts, width, count):
    pairs, seen = [], set()
    for index in np.random.default_rng(5842).permutation(len(cuts)):
        if len(pairs) >= count:
            break
        cut = int(cuts[index])
        start = max(0, cut - width)
        history = bytes(raw[start:cut])
        # The byte window may begin inside a word or a UTF-8 character.
        # Only use complete sentences from the paragraph ending at this cut.
        paragraph_start = max(history.rfind(b'\n'), history.rfind(b'\r')) + 1
        at_paragraph_start = paragraph_start > 0 or start == 0 or raw[start-1] in (10, 13)
        history = history[paragraph_start:]
        starts = [0] if at_paragraph_start else []
        starts.extend(sentence_cuts(history))
        if len(starts) < 2 or starts[-1] - starts[0] < 32:
            continue
        long = history[starts[0]:].decode('utf8').lstrip()
        short = history[starts[-1]:].decode('utf8').lstrip()
        if long in seen:
            continue
        seen.add(long)
        pairs.append(dict(offset=cut, short=short, long=long))
    return pairs

with precision(device):
    raw_passage = sample(prose_model, prompt, SAMPLE_BYTES, SAMPLE_TEMPERATURE, device, SAMPLE_TOP_P,
                         finish_after=max(1, SAMPLE_BYTES//2))
(run_dir/'sample_raw.txt').write_text(raw_passage, encoding='utf8')
passage = display_wiki_sample(raw_passage)
(run_dir/'sample.txt').write_text(passage, encoding='utf8')
if before_passage is not None:
    print('Before this prose run:\n' + display_wiki_sample(before_passage))
print('Selected prose checkpoint:\n' + passage)

pairs = prose_prompt_pairs(corpus['validation'], anchors['validation'], prefix_bytes, PROSE_COMPARISON_PARAGRAPHS)
comparison_bytes = min(240, SAMPLE_BYTES)
comparison = dict(split='validation', checkpoint=lm_results['selected_checkpoint'], seeds=[123, 917],
                  max_new_bytes=comparison_bytes, temperature=SAMPLE_TEMPERATURE, top_p=SAMPLE_TOP_P, pairs=pairs)
with precision(device):
    for i, pair in enumerate(pairs, 1):
        pair['generations'] = []
        for seed in comparison['seeds']:
            completions = {}
            for mode in ('short', 'long'):
                lead = pair[mode]
                generated = sample(prose_model, lead, comparison_bytes, SAMPLE_TEMPERATURE, device,
                                   SAMPLE_TOP_P, seed=seed, finish_after=max(1, comparison_bytes//2))
                completions[mode] = generated[len(lead):]
                print(f'Paragraph {i}, {mode} prompt, seed {seed}\n  Prompt: {display_wiki_sample(lead)}'
                      f'\n  Generated: {display_wiki_sample(completions[mode])}')
            pair['generations'].append(dict(seed=seed, **completions))
if len(pairs) < PROSE_COMPARISON_PARAGRAPHS:
    print(f'Found {len(pairs)} eligible distinct validation paragraphs for the prompt comparison.')
save_json('prose_comparison.json', comparison)

## Optional reasoning probes

Each item has two matched versions and two equal-byte-length candidate endings. In the direct task, swapping where two containers are located changes the correct place. In the two-hop task, locations stay fixed and swapping which container holds an object changes its inferred location. The LM scores every byte of each candidate including its final period. Templates, fact order and vocabulary choices vary; the probe seed stays fixed across runs. The original evaluation items and scoring rule are fixed. The first score is zero-shot. A separate copy of the model then practices both tasks on fresh matched examples. All combinations of objects, containers and places used by the evaluation are excluded from practice, including reordered facts and role swaps. Practice uses the same templates.

For example, suppose a coin is in a box, a key is in a bag, the box is in a barn and the bag is in a shed. The coin is in the barn. Swap which container holds the coin and key, and the answer must change to the shed. **Both-correct pairs** count how often the model gets both versions right. Any context-independent preference for one place gets 50% individual accuracy and zero both-correct pairs. The accuracy intervals resample whole pairs. This probes sensitivity to the supplied facts within these templates.

In [ ]:
def world_key(objects, boxes, places):
    return tuple(tuple(sorted(names)) for names in (objects, boxes, places))

def make_probe(n_pairs=64, seed=719, excluded_worlds=(), template_offset=0, lesson='full', hop=None):
    """Two counterfactual prompts per item; the correct candidate reverses."""
    import random
    rng = random.Random(seed)
    items = []
    seen = set()
    for task in ('direct', 'two_hop'):
        for pair_id in range(n_pairs):
            while True:
                objects = rng.sample(['coin', 'ring', 'key', 'map', 'pen', 'pin', 'bell', 'comb'], 2)
                boxes = rng.sample(['bag', 'box', 'chest', 'case', 'basket', 'drawer'], 2)
                places = rng.sample(['barn', 'cave', 'shed', 'hall', 'mill', 'yurt', 'shop', 'loft'], 2)
                key = (task, *objects, *boxes, *places)
                world = world_key(objects, boxes, places)
                if world not in excluded_worlds and key not in seen:
                    seen.add(key)
                    break
            oi = rng.randrange(2)
            bi = rng.randrange(2)
            # Both queried entities and both fact orders vary independently.
            object_order = rng.sample([0, 1], 2)
            box_order = rng.sample([0, 1], 2)
            template = (pair_id + template_offset) % 2
            for variant in (0, 1):
                box_for = [i ^ oi ^ (variant if task == 'two_hop' else 0) for i in (0, 1)]
                place_for = [i ^ bi ^ (variant if task == 'direct' else 0) for i in (0, 1)]
                facts = []
                for i in object_order:
                    if template == 0:
                        facts.append(f"The {objects[i]} is inside the {boxes[box_for[i]]}.")
                    else:
                        facts.append(f"Inside the {boxes[box_for[i]]} is the {objects[i]}.")
                for i in box_order:
                    if template == 0:
                        facts.append(f"The {boxes[i]} is in the {places[place_for[i]]}.")
                    else:
                        facts.append(f"The {places[place_for[i]]} holds the {boxes[i]}.")
                noun = boxes[0] if task == 'direct' else objects[0]
                answer = place_for[0] if task == 'direct' else place_for[box_for[0]]
                candidates = places
                if lesson == 'one_link' and task == 'two_hop':
                    # Practice the object-to-container link on its own.
                    facts = facts[:2]
                    answer, candidates = box_for[0], boxes
                elif lesson == 'one_link':
                    facts = facts[2:]
                elif lesson == 'support_only':
                    # Both candidate places appear. The object fact determines which box to follow.
                    facts = ([facts[object_order.index(0)]] if task == 'two_hop' else []) + facts[2:]
                item_task = task
                if hop and task == 'two_hop':
                    item_task = hop
                    if hop == 'object_to_container':
                        candidates, answer = boxes, box_for[0]
                    elif hop == 'container_to_place':
                        # Supply the true intermediate box only for this separate diagnostic.
                        noun = boxes[box_for[0]]
                    else:
                        raise ValueError(f'Unknown diagnostic {hop}')
                query = f"The {noun} is therefore in the"
                if task == 'two_hop' and (hop == 'object_to_container' or lesson == 'one_link'):
                    query = f"The {noun} is inside which container? The container is the"
                prompt = ' '.join([*facts, query])
                items.append(dict(task=item_task, pair=pair_id, version=variant, template=template,
                                  world=world, prompt=prompt, endings=[' '+p+'.' for p in candidates], answer=answer))
    return items


def ending_scores(model, items, device):
    # Byte j is predicted at logit position j-1. Include the leading space
    # and final period. The right padding and all prompt targets are unscored.
    sequences, starts, lengths = [], [], []
    for item in items:
        prefix = item['prompt'].encode('utf8')
        for ending in item['endings']:
            suffix = ending.encode('utf8')
            sequences.append(prefix + suffix)
            starts.append(len(prefix)-1)
            lengths.append(len(suffix))
    # Pack on the CPU and upload each buffer once. Compute vocabulary logits only for answer bytes.
    packed = np.zeros((len(sequences), max(map(len, sequences))), dtype=np.uint8)
    for row, seq in enumerate(sequences):
        packed[row, :len(seq)] = np.frombuffer(seq, dtype=np.uint8)
    ids = torch.from_numpy(packed).to(device, non_blocking=True).long()
    columns = np.arange(max(lengths))
    valid = columns < np.asarray(lengths)[:, None]
    positions = np.where(valid, np.asarray(starts)[:, None] + columns, 0)
    positions = torch.from_numpy(positions).to(device, dtype=torch.long, non_blocking=True)
    with precision(device):
        logits = model(ids[:, :-1], positions=positions)
    targets = ids.gather(1, positions+1)
    nll = F.cross_entropy(logits.float().transpose(1, 2), targets, reduction='none')
    return -nll.masked_fill(~torch.from_numpy(valid).to(device, non_blocking=True), 0).sum(1).reshape(-1, 2)


@torch.inference_mode()
def probe_model(model, items, device, sequence_batch=16):
    model.eval()
    rows = []
    item_batch = max(1, sequence_batch//2)
    for first in range(0, len(items), item_batch):
        group = items[first:first+item_batch]
        logps = ending_scores(model, group, device).tolist()
        for i, item in enumerate(group):
            a, b = logps[i]
            left, right = a, b
            unit = 'logp'
            if item['task'] == 'object_to_container':
                # Container names have different byte lengths. Preserve the raw scores for inspection.
                left, right = (score/len(ending.encode('utf8')) for score, ending in zip((a, b), item['endings']))
                unit = 'logp_per_byte'
            predicted = int(right > left)  # exact ties choose the first of the balanced labels
            rows.append(dict(**item, logp_left=a, logp_right=b, predicted=predicted,
                             score_left=left, score_right=right, score_unit=unit,
                             correct=predicted == item['answer'], tie=left == right))
    summary = {}
    for task in dict.fromkeys(row['task'] for row in rows):
        task_rows = [row for row in rows if row['task'] == task]
        choices = np.array([r['predicted'] for r in task_rows]).reshape(-1, 2)
        correct = np.array([r['correct'] for r in task_rows]).reshape(-1, 2)
        rng = np.random.default_rng(917)
        resampled = correct.mean(1)[rng.integers(len(correct), size=(2000, len(correct)))].mean(1)
        summary[task] = dict(prompts=correct.size, pairs=len(correct), correct=int(correct.sum()),
                             accuracy=float(correct.mean()), both_correct=int(correct.all(1).sum()),
                             pair_accuracy=float(correct.all(1).mean()),
                             accuracy_ci95=np.quantile(resampled, [.025,.975]).tolist(),
                             same_choice_pairs=int((choices[:, 0] == choices[:, 1]).sum()),
                             ties=sum(r['tie'] for r in task_rows))
    return summary, rows

def score_hops(model, sequence_batch):
    items = []
    for hop in ('object_to_container', 'container_to_place'):
        items.extend(row for row in make_probe(PROBE_PAIRS, hop=hop) if row['task'] == hop)
    return probe_model(model, items, device, sequence_batch)

In [ ]:
def print_scores(label, summary):
    for task, r in summary.items():
        print(f"{label} {task}: {r['correct']}/{r['prompts']} individual, "
              f"{r['both_correct']}/{r['pairs']} both correct, "
              f"{r['same_choice_pairs']}/{r['pairs']} picked the same candidate twice")

probe_summary = base_hops = None
if RUN_REASONING:
    eval_items = make_probe(PROBE_PAIRS)
    probe_batch = min(32, max(2, batch_size))
    probe_summary, probe_rows = probe_model(prose_model, eval_items, device, probe_batch)
    save_json('reasoning.json', dict(seed=719, stage='zero_shot', summary=probe_summary, items=probe_rows))
    print_scores('Zero-shot', probe_summary)
    base_hops, base_hop_rows = score_hops(prose_model, probe_batch)
    save_json('hops_zero_shot.json', dict(summary=base_hops, items=base_hop_rows))
    print_scores('Zero-shot component', base_hops)

## Practice on separate examples and repeat the probe

Practice alternates short examples with full questions. Short examples teach the object-to-container and container-to-place links, then ask for the final place with the queried object fact and both possible locations. Every batch contains matched swaps. The loss rewards predicting the answer bytes, ranking the correct ending first, and reversing that preference when the facts change. A separate validation set of 16 pairs per task selects the checkpoint with the most both-correct pairs, breaking ties by individual accuracy and then by candidate loss. Update zero is eligible. The original evaluation items are scored again after training. The prose and task checkpoints are saved separately.

In [ ]:
def task_backward(task_model, opt, rng, pairs_per_task, excluded, lesson, template_offset):
    items = make_probe(pairs_per_task, int(rng.integers(2**32)), excluded_worlds=excluded,
                       template_offset=template_offset, lesson=lesson)
    answer_info = [(item['answer'], len(item['endings'][item['answer']].encode('utf8'))) for item in items]
    labels, lengths = torch.tensor(answer_info).to(device, non_blocking=True).unbind(1)
    opt.zero_grad(set_to_none=True)
    scores = ending_scores(task_model, items, device)
    correct = scores.gather(1, labels[:, None]).squeeze(1)
    other = scores.gather(1, (1-labels)[:, None]).squeeze(1)
    # A fixed preference for one noun cancels between the two versions.
    swap_margin = (correct - other).reshape(-1, 2).mean(1)
    loss = F.cross_entropy(scores, labels) - (correct / lengths).mean() + F.softplus(-swap_margin).mean()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(task_model.parameters(), 1., error_if_nonfinite=True)
    return loss.detach()

def practice_tasks(task_model, eval_items):
    excluded = {item['world'] for item in eval_items}
    dev_items = make_probe(16, seed=1831, excluded_worlds=excluded)
    excluded.update(item['world'] for item in dev_items)
    rng = np.random.default_rng(SEED + 1001)
    # Two tasks x two versions x two candidate sequences, each under 192 bytes.
    pairs_per_task = max(1, min(8, batch_size*CONTEXT // (8*192)))
    tuning = None
    if dynamic:
        def measure(pairs):
            trial_model = deepcopy(task_model)
            trial_opt = make_optimizer(trial_model)
            trial_model.train()
            trial_rng = np.random.default_rng(8239)
            def trial_update():
                task_backward(trial_model, trial_opt, trial_rng, pairs, excluded, 'full', 0)
                trial_opt.step()
            seconds, peak = timed_updates(trial_update)
            return dict(batch=pairs, pairs_per_second=2*pairs/seconds,
                        median_train_step_seconds=seconds, peak_allocated_bytes=peak)
        limit = allocation_limit()
        maximum = max(1, MAX_BATCH_BYTES//(8*192))
        winner, trials = choose_batch(measure, pairs_per_task, 1, maximum, limit, 'pairs_per_second')
        pairs_per_task = winner['batch']
        tuning = dict(selected_pairs_per_task=pairs_per_task, memory_limit_bytes=limit, trials=trials)
        print(f'Selected {pairs_per_task} pairs per task per update.')
    pairs_seen = 0
    opt = make_optimizer(task_model)
    updates = 0
    best_key, best_state, best_update = None, None, None

    def validate():
        nonlocal best_key, best_state, best_update
        summary, rows = probe_model(task_model, dev_items, device, probe_batch)
        margins = np.array([(r['logp_right'] - r['logp_left']) * (2*r['answer'] - 1) for r in rows])
        candidate_loss = float(np.logaddexp(0., -margins).mean())
        key = (sum(r['both_correct'] for r in summary.values()),
               sum(r['correct'] for r in summary.values()), -candidate_loss)
        if best_key is None or key > best_key:
            best_key, best_update = key, updates
            best_state = {k: v.to('cpu', copy=True) for k, v in task_model.state_dict().items()}
        event(stage='task_validation', update=updates, direct_both=summary['direct']['both_correct'],
              two_hop_both=summary['two_hop']['both_correct'], pairs_per_task=16, candidate_loss=candidate_loss)
        task_model.train()

    validate()
    if DEVICE == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
    start = time.perf_counter()
    last_check, checked_update = start, 0
    while time.perf_counter() - start < 60*TASK_MINUTES and (MAX_UPDATES is None or updates < MAX_UPDATES):
        lesson = ('support_only', 'one_link', 'full', 'full')[updates % 4]
        def backward(pairs):
            return task_backward(task_model, opt, rng, pairs, excluded, lesson, (updates//4 + updates) % 2)
        loss, pairs_per_task = backward_with_backoff(backward, opt, rng, pairs_per_task, 1, dynamic)
        opt.step()
        updates += 1
        pairs_seen += pairs_per_task
        if updates == 1 or updates % 100 == 0:
            event(stage='task_train', update=updates, loss=float(loss), pairs_per_task_per_update=pairs_per_task,
                  loop_pairs_per_second=2*pairs_seen/(time.perf_counter()-start), **gpu_stats())
        if time.perf_counter() - last_check >= 90:
            validate()
            last_check, checked_update = time.perf_counter(), updates
    if updates != checked_update:
        validate()
    task_model.load_state_dict(best_state)
    task_model.eval()
    sync()
    return dict(updates=updates, pairs_per_task=pairs_seen, tuning=tuning, seconds=time.perf_counter()-start,
                selected_update=best_update, dev_seed=1831, seed=SEED + 1001, lr=LR, excluded_worlds=len(excluded))

task_result = None
if RUN_REASONING and TASK_MINUTES > 0:
    task_model = deepcopy(prose_model)
    practice = practice_tasks(task_model, eval_items)
    print(f"Validation selected task update {practice['selected_update']} of {practice['updates']}.")
    tuned_summary, tuned_rows = probe_model(task_model, eval_items, device, probe_batch)
    task_path = run_dir/'task_tuned.pt'
    torch.save(dict(kind='C1', slots=8, state_dict=task_model.state_dict(), practice=practice,
                    base_step=best_step), task_path)
    tuned_hops, tuned_hop_rows = score_hops(task_model, probe_batch)
    save_json('hops_tuned.json', dict(summary=tuned_hops, items=tuned_hop_rows))
    print_scores('After practice component', tuned_hops)
    task_result = dict(practice=practice, summary=tuned_summary, components=tuned_hops, checkpoint=str(task_path))
    save_json('reasoning_tuned.json', dict(**task_result, seed=719, items=tuned_rows))
    print_scores('After practice', tuned_summary)

report = dict(config=config, hardware=hardware, benchmark=bench, prose_tuning=prose_tuning, language_model=lm_results,
              prose_comparison_file="prose_comparison.json", reasoning=probe_summary, components_zero_shot=base_hops, reasoning_tuned=task_result, checkpoint=str(resume_checkpoint) if resume_checkpoint else None,
              sample=dict(temperature=SAMPLE_TEMPERATURE, top_p=SAMPLE_TOP_P, seed=123, prompt=prompt,
                          bytes=SAMPLE_BYTES, generated_bytes=len(raw_passage.encode())-len(prompt.encode()),
                          finish_after=max(1, SAMPLE_BYTES//2), checkpoint=str(run_dir/'best_lm.pt')))
save_json('report.json', report)
print('Results and checkpoints:', run_dir)

### Reading the result

Validation BPB measures next-byte prediction on Wikipedia prose. Held-out continuation matching tests whether the model gives the same continuation higher probability after its real paragraph. Chance matching is 50%. Read the actual passages to judge coherence. The component probes ask for the container and for the place given the true container. Container scores are averaged per byte because names vary in length. Place candidates have equal lengths. The end-to-end question still requires both links. Compare the two probe stages using **both versions correct** and **same candidate twice**. A fixed preference can give half the individual answers correct. The supervised result measures transfer to new combinations within the practiced templates.

## Fact-to-answer learning, sustained prose and conversation

This section runs in two cleanly separated stages:

1. **Counterfactual fact-to-answer learning (`FACT_SWITCH_MINUTES`)**: Uses strictly disjoint OOV evaluation vocabularies and 2×2 counterfactual quads (value-swap + question-swap) with anti-memorization nonce words. Stage 1 (`plain_nll`) builds the multi-byte copy circuit; Stage 2 (`contrastive`) drives key-matched disambiguation without letting Sentence-1 positional shortcuts pass. Checkpoints are selected by counterfactual pair accuracy and greedy exact match (never average BPB), with causal Tree ablations (`on`, `off`, `shuffle`, and `counterfactual_tree_swap`). Optionally set `RUN_ARCHITECTURE_SHOWDOWN = True` on an A100 to compare `C1(L7)`, `C1(L3+L7)`, native `C1(rw=64 Tree)`, and a 10M full Transformer (`A0`) under identical step and quad budgets.
2. **Rebalanced grounded QA, sustained prose and service chat (`DIALOGUE_MINUTES`)**: Separates grounded QA (`Passage:/Question:/Answer:` when `QA_PROMPT_STYLE = 'qa'`) from Taskmaster service chat (`User:/Assistant:`) to eliminate cross-domain style leakage, auto-tunes CUDA batch size even when `TRAIN_MINUTES = 0`, rebalances grounded QA / extractive QA / counterfactual quads / multi-sentence `prose_long` continuations / WikiText anchors, and selects checkpoints using both validation BPB and live counterfactual fact-switching. [CoQA](https://stanfordnlp.github.io/coqa/) · [Taskmaster-1](https://github.com/google-research-datasets/Taskmaster/tree/master/TM-1-2019)

In [ ]:
"""Prepare disjoint prose continuations, grounded QA turns and assistant replies."""
import hashlib
import json
import re
from pathlib import Path
from urllib.request import urlopen

import torch


SOURCES = {
    'coqa-train-v1.0.json': 'https://downloads.cs.stanford.edu/nlp/data/coqa/coqa-train-v1.0.json',
    'coqa-dev-v1.0.json': 'https://downloads.cs.stanford.edu/nlp/data/coqa/coqa-dev-v1.0.json',
    'self-dialogs.json': (
        'https://raw.githubusercontent.com/google-research-datasets/Taskmaster/'
        'd92cb6af3005f1dc09c39e75e7daf4a04905e00b/TM-1-2019/self-dialogs.json'
    ),
}


def read_dialogue_data(directory, offline):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    data, hashes = {}, {}
    for name, url in SOURCES.items():
        path = directory / name
        if not path.is_file() and not offline:
            with urlopen(url, timeout=120) as response:
                raw = response.read()
            json.loads(raw)  # Validate before publishing a cached download.
            tmp = path.with_suffix('.partial')
            tmp.write_bytes(raw)
            tmp.replace(path)
        raw = path.read_bytes()
        data[name] = json.loads(raw)
        hashes[name] = hashlib.sha256(raw).hexdigest()
    return data, hashes


def single_line(text):
    return ' '.join(text.split())


def role_labels(passage, qa_style='role'):
    """Use distinct Question/Answer cues for grounded QA when qa_style='qa' to avoid chat leakage."""
    if passage and qa_style == 'qa':
        return 'Question', 'Answer'
    if qa_style not in ('role', 'qa'):
        raise ValueError(f'Unknown qa_style: {qa_style}')
    return 'User', 'Assistant'


def chat_prompt(user, history, passage, budget, qa_style='role'):
    """Keep the passage, current question and as many complete recent turns as fit."""
    u_tag, a_tag = role_labels(passage, qa_style)
    lead = 'Passage: ' + single_line(passage) + '\n' if passage else ''
    tail = f'{u_tag}: {single_line(user)}\n{a_tag}: '
    room = budget - len((lead + tail).encode())
    if room < 0:
        return None
    for question, answer in reversed(history):
        turn = f'{u_tag}: {single_line(question)}\n{a_tag}: {single_line(answer)}\n'
        size = len(turn.encode())
        if size > room:
            break
        tail, room = turn + tail, room - size
    return lead + tail


def story_continuations(story, context):
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z"“])', story)
    for i in range(1, len(sentences)):
        prompt = ' '.join(sentences[max(0, i-2):i]) + ' '
        target = ' '.join(sentences[i:i+2])
        if len(target.encode()) > 224:
            target = sentences[i]
        if len(target.encode()) <= 224 and len((prompt + target).encode()) <= context:
            yield prompt, target


def long_story_continuations(story, context, min_prefix_bytes=220, target_bytes=280):
    """Multi-sentence prefixes (up to context - target_bytes) for practicing sustained prose."""
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z"“])', story)
    prefix_budget = context - target_bytes
    for i in range(3, len(sentences)):
        target = ' '.join(sentences[i:i+3])
        if len(target.encode()) > target_bytes:
            target = ' '.join(sentences[i:i+2])
        if len(target.encode()) > target_bytes:
            target = sentences[i]
        if not target or len(target.encode()) > target_bytes:
            continue
        # Grow backwards from sentence i-1 so the prompt keeps the fullest recent narrative arc.
        kept, used = [], 0
        for sent in reversed(sentences[:i]):
            add = len(sent.encode()) + 1
            if used + add > prefix_budget:
                break
            kept.append(sent)
            used += add
        if used < min_prefix_bytes or len(kept) < 3:
            continue
        prompt = ' '.join(reversed(kept)) + ' '
        if len((prompt + target).encode()) <= context:
            yield prompt, target


def training_excerpt(story, answer):
    """Use an annotated evidence span only for short, extractive training questions."""
    start, end = answer.get('span_start', -1), answer.get('span_end', -1)
    reply = single_line(answer['input_text']).casefold()
    if not (0 <= start < end <= len(story)) or reply in ('', 'yes', 'no', 'unknown'):
        return ''
    left = max(0, start-100)
    right = min(len(story), end+100)
    while left and not story[left-1].isspace():
        left -= 1
    while right < len(story) and not story[right].isspace():
        right += 1
    excerpt = single_line(story[left:right])
    return excerpt if reply in excerpt.casefold() else ''


def is_extractive_qa(prompt, target, max_answer_bytes=48):
    """True for non-yes/no/unknown answers whose normalized text appears inside the prompt passage."""
    ans = single_line(target).casefold()
    if not ans or ans in ('yes', 'no', 'unknown') or len(ans.encode()) > max_answer_bytes:
        return False
    passage = prompt.split('\n', 1)[0].casefold()
    return ans in passage


def prepare_dialogues(data, context, reply_bytes=160, qa_style='role'):
    splits = {split: {task: [] for task in ('prose', 'qa', 'chat')}
              for split in ('train', 'validation', 'heldout')}
    prompt_budget = context - reply_bytes
    splits['train']['qa_excerpt'] = []
    splits['train']['qa_extractive'] = []
    splits['train']['prose_long'] = []

    def group(text):
        return int.from_bytes(hashlib.sha256(text.encode()).digest()[:4], 'big') % 100

    def add_reply(rows, user, reply, history, passage, style='role'):
        prompt = chat_prompt(user, history, passage, prompt_budget, qa_style=style)
        target = reply + '\n'
        if prompt is not None and reply and len(target.encode()) <= reply_bytes:
            rows.append((prompt, target))
            return prompt, target
        return None

    # Read the official development set first so duplicated passages stay reserved.
    seen_stories = set()
    for name in ('coqa-dev-v1.0.json', 'coqa-train-v1.0.json'):
        for item in data[name]['data']:
            story = single_line(item['story'])
            if not story or story in seen_stories:
                continue
            seen_stories.add(story)
            split = 'heldout' if name.startswith('coqa-dev') else (
                'validation' if group(story) < 5 else 'train')
            pools = splits[split]
            pools['prose'].extend(story_continuations(story, context))
            if split == 'train':
                pools['prose_long'].extend(long_story_continuations(story, context))
            history = []
            for q, a in zip(item['questions'], item['answers'], strict=True):
                if q['turn_id'] != a['turn_id']:
                    raise ValueError('Misaligned CoQA turn')
                question, answer = single_line(q['input_text']), single_line(a['input_text'])
                added = add_reply(pools['qa'], question, answer, history, story, style=qa_style)
                if split == 'train':
                    if added is not None and is_extractive_qa(*added):
                        pools['qa_extractive'].append(added)
                    excerpt = training_excerpt(item['story'], a)
                    if excerpt:
                        add_reply(pools['qa_excerpt'], question, answer, history[-1:], excerpt, style=qa_style)
                history.append((question, answer))

    for item in data['self-dialogs.json']:
        turns = []
        for utterance in item['utterances']:
            speaker, text = utterance['speaker'], single_line(utterance['text'])
            if not text:
                continue
            if turns and speaker == turns[-1][0]:
                turns[-1] = (speaker, turns[-1][1] + ' ' + text)
            else:
                turns.append((speaker, text))
        bucket = group(json.dumps(turns, ensure_ascii=False))
        split = 'heldout' if bucket < 5 else 'validation' if bucket < 10 else 'train'
        history = []
        for (speaker, question), (next_speaker, answer) in zip(turns, turns[1:]):
            if (speaker, next_speaker) == ('USER', 'ASSISTANT'):
                add_reply(splits[split]['chat'], question, answer, history, '', style='role')
                history.append((question, answer))
    return splits


def target_batch(rows, device, history_targets=False):
    """Mask prompts and padding. Training can also supervise preceding assistant/answer turns."""
    encoded = [(prompt.encode(), target.encode()) for prompt, target in rows]
    length = max(len(prompt) + len(target) - 1 for prompt, target in encoded)
    x = torch.zeros((len(rows), length), dtype=torch.long)
    y = torch.full_like(x, -100)
    for i, (prompt, target) in enumerate(encoded):
        tokens = torch.tensor(list(prompt + target))
        end = len(tokens) - 1
        start = len(prompt) - 1
        x[i, :end] = tokens[:-1]
        if history_targets:
            for reply in re.finditer(rb'(?m)^(?:Assistant|Answer): ([^\n]*\n)', prompt):
                first, last = reply.span(1)
                y[i, first-1:last-1] = tokens[first:last]
        y[i, start:end] = tokens[start+1:]
    return x.to(device), y.to(device)

In [ ]:
#@title Counterfactual fact-to-answer benchmark and trainer
"""Counterfactual fact-to-answer benchmark, 2x2 quad trainer, and Tree causal ablation suite.

Designed to run both as a standalone Python module/CLI and embedded verbatim inside run.ipynb
on Google Colab (where LM and dialogue helpers are already defined in notebook globals).
"""
from __future__ import annotations

import argparse
from collections import defaultdict
from contextlib import nullcontext
import json
import math
from pathlib import Path
import random
import re
import time

import numpy as np
import torch
from torch.nn import functional as F

if 'chat_prompt' not in globals():
    from dialogue import chat_prompt, prepare_dialogues, read_dialogue_data, single_line, target_batch
if 'LM' not in globals():
    from model import LM


# Broad training entity vocabulary — strictly disjoint from OOV_VOCAB and user_audit_pairs().
TRAIN_VOCAB = {
    'people': [
        'Liam', 'Noah', 'Nora', 'Maya', 'Omar', 'Sara', 'Paul', 'Ruth',
        'Carl', 'Leah', 'Owen', 'Iris', 'Hugo', 'Vera', 'Adam', 'Cora',
        'Alan', 'Beth', 'Cole', 'Dana', 'Eric', 'Faye', 'Gabe', 'Hana',
        'Ivan', 'Jade', 'Karl', 'Lola', 'Marc', 'Nadia', 'Otto', 'Rosa',
        'Seth', 'Tara', 'Umar', 'Veda', 'Wade', 'Yara', 'Zane', 'Alec',
        'Bria', 'Cleo', 'Dean', 'Elia', 'Finn', 'Gina', 'Hank', 'Inez',
        'Joel', 'Kira', 'Leon', 'Mona', 'Neil', 'Olga', 'Pete', 'Rhea',
        'Saul', 'Thea', 'Vince', 'Wren', 'Bram', 'Dora', 'Emil', 'Freya',
    ],
    'colors': [
        'red', 'yellow', 'black', 'white', 'brown', 'silver', 'gold', 'purple',
        'orange', 'amber', 'ivory', 'scarlet', 'teal', 'coral', 'maroon', 'olive',
        'beige', 'bronze', 'copper', 'indigo', 'jade', 'khaki', 'lilac', 'ochre',
        'peach', 'plum', 'ruby', 'rust', 'sage', 'tan', 'taupe', 'azure',
        'cyan', 'ebony', 'hazel', 'lemon', 'mauve', 'mint', 'navy', 'pearl',
    ],
    'objects': [
        'jacket', 'bottle', 'camera', 'ticket', 'letter', 'wallet', 'helmet', 'mirror',
        'candle', 'hammer', 'guitar', 'parcel', 'lantern', 'compass', 'blanket', 'kettle',
        'ribbon', 'scarf', 'shovel', 'teapot', 'violin', 'whistle', 'wrench', 'apron',
        'badge', 'basket', 'beaker', 'binder', 'buckle', 'button', 'carpet', 'chisel',
        'cloak', 'collar', 'crayon', 'dagger', 'flute', 'funnel', 'gloves', 'goblet',
        'handle', 'hatchet', 'locket', 'magnet', 'mallet', 'mitten', 'muffler', 'napkin',
        'paddle', 'pebble', 'pencil', 'pepper', 'pillow', 'pitcher', 'plaque', 'pocket',
        'poster', 'pouch', 'pulley', 'quiver', 'racket', 'rattle', 'saddle', 'sandal',
    ],
    'places': [
        'kitchen', 'garden', 'station', 'museum', 'harbor', 'bakery', 'office', 'clinic',
        'market', 'theater', 'cellar', 'studio', 'archive', 'attic', 'balcony', 'barn',
        'bazaar', 'boathouse', 'brewery', 'canteen', 'castle', 'cavern', 'closet', 'courtyard',
        'depot', 'dormitory', 'factory', 'foundry', 'gallery', 'greenhouse', 'hangar', 'hostel',
        'kennel', 'laundry', 'lobby', 'lounge', 'mill', 'nursery', 'orchard', 'parlor',
        'pavilion', 'pharmacy', 'pier', 'plaza', 'porch', 'quarry', 'shed', 'stable',
        'tavern', 'terrace', 'tower', 'tunnel', 'vault', 'veranda', 'warehouse', 'workshop',
    ],
    'cities': [
        'Paris', 'Rome', 'Cairo', 'Tokyo', 'Seoul', 'Prague', 'Vienna', 'Dublin',
        'Lisbon', 'Athens', 'Madrid', 'Zurich', 'Accra', 'Baku', 'Basel', 'Bilbao',
        'Bonn', 'Brest', 'Bruges', 'Cadiz', 'Cannes', 'Cebu', 'Cusco', 'Dakar',
        'Doha', 'Essen', 'Faro', 'Genoa', 'Ghent', 'Graz', 'Hanoi', 'Hull',
        'Izmir', 'Jena', 'Kobe', 'Krakow', 'Kyoto', 'Lagos', 'Leeds', 'Lille',
        'Linz', 'Lucca', 'Malmo', 'Milan', 'Minsk', 'Modena', 'Munich', 'Nantes',
        'Naples', 'Nice', 'Osaka', 'Padua', 'Pisa', 'Porto', 'Quito', 'Rabat',
        'Reno', 'Riga', 'Rouen', 'Siena', 'Sofia', 'Split', 'Suva', 'Tallinn',
        'Tampa', 'Trier', 'Tunis', 'Turin', 'Ulm', 'Vigo', 'Warsaw', 'York',
    ],
    'days': ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Saturday'],
    'events': [
        'concert', 'lecture', 'briefing', 'workshop', 'hearing', 'summit',
        'auction', 'ballet', 'contest', 'debate', 'exhibit', 'festival',
        'gala', 'match', 'opera', 'parade', 'rally', 'recital',
        'rehearsal', 'review', 'show', 'speech', 'symposium', 'trial',
    ],
}

OOV_VOCAB = {
    'people': ['Mira', 'Anna', 'Emma', 'Luca', 'Nina', 'Ezra', 'Tessa', 'Jonas'],
    'colors': ['blue', 'pink', 'green', 'grey', 'violet', 'crimson'],
    'objects': ['notebook', 'folder', 'atlas', 'map', 'journal', 'satchel'],
    'places': ['library', 'cabin', 'chapel', 'garage', 'temple', 'pantry'],
    'cities': ['Lima', 'Oslo', 'Bern', 'Lyon', 'Cork', 'Perth'],
    'days': ['Friday', 'Sunday'],
    'events': ['meeting', 'dinner', 'seminar', 'forum', 'session', 'banquet'],
}

_OOV_FORBIDDEN = {w.casefold() for vals in OOV_VOCAB.values() for w in vals} | {'desk', 'bag'}

FILLER_SENTENCES = [
    'The weather stayed mild throughout the afternoon while trains ran on schedule.',
    'Several clerks checked the roster near the front entrance before closing time.',
    'A quiet breeze moved across the courtyard as streetlights flickered along the lane.',
    'Two delivery vans parked outside the west gate and unloaded wooden crates.',
    'The corridor remained calm except for footsteps echoing from the upper stairs.',
    'Rain clouds gathered above the ridge, though no storm arrived before evening.',
]

_CONSONANTS = 'bcdfghjklmnprstvwz'
_VOWELS = 'aeiou'

ALL_FAMILIES = ('city', 'place', 'object', 'color_fwd', 'color', 'person')
FAMILY_DIRECTION = {
    'city': 'forward',
    'place': 'forward',
    'object': 'forward',
    'color_fwd': 'forward',
    'color': 'reverse',
    'person': 'reverse',
}


def random_nonce(rng: random.Random, capitalize: bool = False) -> str:
    """Generate a pronounceable novel 4-7 byte word guaranteed outside OOV_VOCAB."""
    while True:
        length = rng.choice((4, 5, 6, 7))
        chars = []
        start_vowel = rng.random() < 0.25
        for i in range(length):
            use_vowel = ((i % 2 == 0) == start_vowel)
            chars.append(rng.choice(_VOWELS if use_vowel else _CONSONANTS))
        word = ''.join(chars)
        if word.casefold() not in _OOV_FORBIDDEN:
            return word.capitalize() if capitalize else word


def user_audit_pairs(qa_style: str = 'role') -> list[dict]:
    """The six diagnostic pairs (clean + distractor) matching the reported failure cases."""
    specs = [
        (
            'color_clean', 'color', False,
            'Mira left her {a} notebook at the library.',
            "What color was Mira's notebook?",
            'blue', 'pink',
        ),
        (
            'color_dist', 'color', True,
            'Mira left her {a} notebook at the library. She put the {b} folder in her bag.',
            "What color was Mira's notebook?",
            'blue', 'pink',
        ),
        (
            'city_clean', 'city', False,
            'The meeting will be held in {a} on Friday.',
            'Where will the meeting be held?',
            'Lima', 'Oslo',
        ),
        (
            'city_dist', 'city', True,
            'The meeting will be held in {a} on Friday. The dinner will be in {b}.',
            'Where will the meeting be held?',
            'Lima', 'Oslo',
        ),
        (
            'person_clean', 'person', False,
            '{a} borrowed the green atlas from the desk.',
            'Who borrowed the atlas?',
            'Anna', 'Emma',
        ),
        (
            'person_dist', 'person', True,
            '{a} borrowed the green atlas from the desk. {b} took the grey map.',
            'Who borrowed the atlas?',
            'Anna', 'Emma',
        ),
    ]
    pairs = []
    for pid, kind, has_dist, tmpl, q, va, vb in specs:
        pa = chat_prompt(q, [], tmpl.format(a=va, b=vb), 1400, qa_style=qa_style)
        pb = chat_prompt(q, [], tmpl.format(a=vb, b=va), 1400, qa_style=qa_style)
        pairs.append(dict(
            id=pid, family=kind, direction=FAMILY_DIRECTION[kind], distractor=has_dist,
            prompt_a=pa, prompt_b=pb, answer_a=va, answer_b=vb,
        ))
    return pairs


def user_qswap_pairs(qa_style: str = 'role') -> list[dict]:
    """Question-swap counterparts of the user's distractor passages (cancels S1/S2 shortcuts)."""
    specs = [
        (
            'color_qswap', 'color', 'reverse',
            'Mira left her blue notebook at the library. Mira left her pink folder in her bag.',
            "What color was Mira's notebook?", "What color was Mira's folder?",
            'blue', 'pink',
        ),
        (
            'city_qswap', 'city', 'forward',
            'The meeting will be held in Lima on Friday. The dinner will be held in Oslo on Sunday.',
            'Where will the meeting be held?', 'Where will the dinner be held?',
            'Lima', 'Oslo',
        ),
        (
            'person_qswap', 'person', 'reverse',
            'Anna borrowed the green atlas from the desk. Emma borrowed the grey map from the desk.',
            'Who borrowed the atlas?', 'Who borrowed the map?',
            'Anna', 'Emma',
        ),
    ]
    return [
        dict(
            id=pid, family=fam, direction=direction, distractor=True,
            prompt_a=chat_prompt(qa, [], passage, 1400, qa_style=qa_style),
            prompt_b=chat_prompt(qb, [], passage, 1400, qa_style=qa_style),
            answer_a=va, answer_b=vb,
        )
        for pid, fam, direction, passage, qa, qb, va, vb in specs
    ]


def _draw_distinct_pair(rng: random.Random, pool: list[str], nonce_prob: float, capitalize: bool) -> tuple[str, str]:
    if nonce_prob > 0 and rng.random() < nonce_prob:
        a = random_nonce(rng, capitalize=capitalize)
        b = random_nonce(rng, capitalize=capitalize)
        while b.casefold() == a.casefold() or b[0] == a[0]:
            b = random_nonce(rng, capitalize=capitalize)
        return a, b
    a, b = rng.sample(pool, 2)
    tries = 0
    while b[0] == a[0] and tries < 12:
        a, b = rng.sample(pool, 2)
        tries += 1
    return a, b


def _build_pair(
    rng: random.Random,
    vocab: dict[str, list[str]],
    family: str,
    distractor: bool,
    unseen_template: bool,
    filler_count: int,
    pair_id: str,
    *,
    nonce_value_prob: float = 0.0,
    nonce_key_prob: float = 0.0,
    qa_style: str = 'role',
) -> dict:
    """Build a strictly symmetric counterfactual value-swap pair."""
    p1, p2 = _draw_distinct_pair(rng, vocab['people'], nonce_value_prob if family == 'person' else nonce_key_prob, True)
    c1, c2 = _draw_distinct_pair(rng, vocab['colors'], nonce_value_prob if family in ('color', 'color_fwd') else 0.0, False)
    o1, o2 = _draw_distinct_pair(rng, vocab['objects'], nonce_value_prob if family == 'object' else nonce_key_prob, False)
    l1, l2 = _draw_distinct_pair(rng, vocab['places'], nonce_value_prob if family == 'place' else 0.0, False)
    t1, t2 = _draw_distinct_pair(rng, vocab['cities'], nonce_value_prob if family == 'city' else 0.0, True)
    e1, e2 = _draw_distinct_pair(rng, vocab['events'], nonce_key_prob if family == 'city' else 0.0, False)
    day1, day2 = rng.sample(vocab['days'], 2)

    if family == 'color':
        va, vb = c1, c2
        if not unseen_template:
            tpl = rng.randrange(3)
            if tpl == 0:
                s1 = lambda v: f'{p1} left the {v} {o1} at the {l1}.'
                s2 = lambda v: f'{p1} left the {v} {o2} at the {l2}.'
                q = f"What color was {p1}'s {o1}?"
            elif tpl == 1:
                s1 = lambda v: f'{p1} bought a {v} {o1} on {day1}.'
                s2 = lambda v: f'{p1} bought a {v} {o2} on {day2}.'
                q = f'What color is the {o1}?'
            else:
                s1 = lambda v: f'The {v} {o1} sat at the {l1}.'
                s2 = lambda v: f'The {v} {o2} sat at the {l2}.'
                q = f'What color was the {o1}?'
        else:
            s1 = lambda v: f'Inside the {l1}, {p1} noticed a {v} {o1}.'
            s2 = lambda v: f'Inside the {l2}, {p1} noticed a {v} {o2}.'
            q = f'Which color described the {o1}?'
    elif family == 'color_fwd':
        va, vb = c1, c2
        if not unseen_template:
            tpl = rng.randrange(2)
            if tpl == 0:
                s1 = lambda v: f'The {o1} on the table was {v}.'
                s2 = lambda v: f'The {o2} on the table was {v}.'
                q = f'What color was the {o1}?'
            else:
                s1 = lambda v: f'{p1} chose a {v} {o1}.'
                s2 = lambda v: f'{p2} chose a {v} {o1}.'
                q = f"What color was {p1}'s {o1}?"
        else:
            s1 = lambda v: f'The paint on the {o1} looked {v}.'
            s2 = lambda v: f'The paint on the {o2} looked {v}.'
            q = f'Which color was the {o1}?'
    elif family == 'place':
        va, vb = l1, l2
        if not unseen_template:
            tpl = rng.randrange(3)
            if tpl == 0:
                s1 = lambda v: f'{p1} left the {o1} at the {v}.'
                s2 = lambda v: f'{p2} left the {o2} at the {v}.'
                q = f'Where did {p1} leave the {o1}?'
            elif tpl == 1:
                s1 = lambda v: f'The {c1} {o1} is stored in the {v}.'
                s2 = lambda v: f'The {c2} {o2} is stored in the {v}.'
                q = f'Where is the {o1} stored?'
            else:
                s1 = lambda v: f'{p1} found the {o1} inside the {v}.'
                s2 = lambda v: f'{p2} found the {o2} inside the {v}.'
                q = f'Where did {p1} find the {o1}?'
        else:
            s1 = lambda v: f'According to the note, the {o1} remained at the {v}.'
            s2 = lambda v: f'According to the note, the {o2} remained at the {v}.'
            q = f'At which location did the {o1} remain?'
    elif family == 'city':
        va, vb = t1, t2
        if not unseen_template:
            tpl = rng.randrange(3)
            if tpl == 0:
                s1 = lambda v: f'The {e1} will be held in {v} on {day1}.'
                s2 = lambda v: f'The {e2} will be held in {v} on {day2}.'
                q = f'Where will the {e1} be held?'
            elif tpl == 1:
                s1 = lambda v: f'{p1} traveled to {v} for the {e1}.'
                s2 = lambda v: f'{p2} traveled to {v} for the {e2}.'
                q = f'Which city did {p1} travel to?'
            else:
                s1 = lambda v: f'The {e1} opens in {v} on {day1}.'
                s2 = lambda v: f'The {e2} opens in {v} on {day2}.'
                q = f'Where does the {e1} open?'
        else:
            s1 = lambda v: f'Organizers scheduled the {e1} in {v}.'
            s2 = lambda v: f'Organizers scheduled the {e2} in {v}.'
            q = f'Which city is hosting the {e1}?'
    elif family == 'person':
        va, vb = p1, p2
        if not unseen_template:
            tpl = rng.randrange(3)
            if tpl == 0:
                s1 = lambda v: f'{v} borrowed the {c1} {o1} from the {l1}.'
                s2 = lambda v: f'{v} borrowed the {c2} {o2} from the {l2}.'
                q = f'Who borrowed the {o1}?'
            elif tpl == 1:
                s1 = lambda v: f'{v} brought the {o1} to the {e1}.'
                s2 = lambda v: f'{v} brought the {o2} to the {e2}.'
                q = f'Who brought the {o1}?'
            else:
                s1 = lambda v: f'{v} repaired the {c1} {o1} on {day1}.'
                s2 = lambda v: f'{v} repaired the {c2} {o2} on {day2}.'
                q = f'Who repaired the {o1}?'
        else:
            s1 = lambda v: f'The {c1} {o1} was returned to the {l1} by {v}.'
            s2 = lambda v: f'The {c2} {o2} was returned to the {l2} by {v}.'
            q = f'Who returned the {o1}?'
    elif family == 'object':
        va, vb = o1, o2
        if not unseen_template:
            tpl = rng.randrange(2)
            if tpl == 0:
                s1 = lambda v: f'{p1} carried a {v} into the {l1}.'
                s2 = lambda v: f'{p2} carried a {v} into the {l2}.'
                q = f'What did {p1} carry?'
            else:
                s1 = lambda v: f'{p1} packed a {v} before leaving for {t1}.'
                s2 = lambda v: f'{p2} packed a {v} before leaving for {t2}.'
                q = f'What did {p1} pack?'
        else:
            s1 = lambda v: f'Before noon, {p1} placed a {v} inside the {l1}.'
            s2 = lambda v: f'Before noon, {p2} placed a {v} inside the {l2}.'
            q = f'Which item did {p1} place inside the {l1}?'
    else:
        raise ValueError(f'Unknown family: {family}')

    flip_order = distractor and (rng.randrange(2) == 1)
    fillers = rng.sample(FILLER_SENTENCES, filler_count) if filler_count else []

    def compose(main_val: str, dist_val: str) -> str:
        sents = (
            [s2(dist_val), s1(main_val)] if flip_order
            else ([s1(main_val), s2(dist_val)] if distractor else [s1(main_val)])
        )
        sents.extend(fillers)
        return ' '.join(sents)

    pass_a = compose(va, vb)
    pass_b = compose(vb, va)
    return dict(
        id=pair_id,
        family=family,
        direction=FAMILY_DIRECTION[family],
        distractor=distractor,
        filler_count=filler_count,
        unseen_template=unseen_template,
        prompt_a=chat_prompt(q, [], pass_a, 1400, qa_style=qa_style),
        prompt_b=chat_prompt(q, [], pass_b, 1400, qa_style=qa_style),
        answer_a=va,
        answer_b=vb,
    )


def make_quad_as_two_pairs(
    rng: random.Random,
    vocab: dict[str, list[str]],
    family: str = 'city',
    *,
    nonce_val_prob: float = 0.5,
    nonce_key_prob: float = 0.35,
    filler_count: int = 0,
    qa_style: str = 'role',
) -> list[dict]:
    """Build a 2x2 counterfactual quad as two question-swap pairs with swapped values.

    Each pair holds the passage fixed while swapping which entity is queried (cancels S1/S2
    positional shortcuts). Across the two pairs, values swap (cancels static word priors).
    """
    p1, p2 = _draw_distinct_pair(rng, vocab['people'], nonce_val_prob if family == 'person' else nonce_key_prob, True)
    c1, c2 = _draw_distinct_pair(rng, vocab['colors'], nonce_val_prob if family in ('color', 'color_fwd') else 0.0, False)
    o1, o2 = _draw_distinct_pair(rng, vocab['objects'], nonce_val_prob if family == 'object' else nonce_key_prob, False)
    l1, l2 = _draw_distinct_pair(rng, vocab['places'], nonce_val_prob if family == 'place' else 0.0, False)
    t1, t2 = _draw_distinct_pair(rng, vocab['cities'], nonce_val_prob if family == 'city' else 0.0, True)
    e1, e2 = _draw_distinct_pair(rng, vocab['events'], nonce_key_prob if family == 'city' else 0.0, False)
    d1, d2 = rng.sample(vocab['days'], 2)

    if family == 'city':
        va, vb = t1, t2
        s = lambda a, b: f'The {e1} will be held in {a} on {d1}. The {e2} will be held in {b} on {d2}.'
        qa, qb = f'Where will the {e1} be held?', f'Where will the {e2} be held?'
    elif family == 'color':
        va, vb = c1, c2
        s = lambda a, b: f'{p1} left her {a} {o1} at the {l1}. {p1} left her {b} {o2} at the {l2}.'
        qa, qb = f"What color was {p1}'s {o1}?", f"What color was {p1}'s {o2}?"
    elif family == 'place':
        va, vb = l1, l2
        s = lambda a, b: f'{p1} left the {o1} at the {a}. {p1} left the {o2} at the {b}.'
        qa, qb = f'Where did {p1} leave the {o1}?', f'Where did {p1} leave the {o2}?'
    elif family == 'person':
        va, vb = p1, p2
        s = lambda a, b: f'{a} borrowed the {c1} {o1} from the {l1}. {b} borrowed the {c2} {o2} from the {l2}.'
        qa, qb = f'Who borrowed the {o1}?', f'Who borrowed the {o2}?'
    elif family == 'object':
        va, vb = o1, o2
        s = lambda a, b: f'{p1} carried a {a} into the {l1}. {p2} carried a {b} into the {l2}.'
        qa, qb = f'What did {p1} carry?', f'What did {p2} carry?'
    elif family == 'color_fwd':
        va, vb = c1, c2
        s = lambda a, b: f'The {o1} on the table was {a}. The {o2} on the table was {b}.'
        qa, qb = f'What color was the {o1}?', f'What color was the {o2}?'
    else:
        raise ValueError(f'Unknown family: {family}')

    tail = (' ' + ' '.join(rng.sample(FILLER_SENTENCES, filler_count))) if filler_count else ''
    p_ab = s(va, vb) + tail
    p_ba = s(vb, va) + tail
    direction = FAMILY_DIRECTION[family]
    return [
        dict(
            id='q1', family=family, direction=direction, distractor=True,
            prompt_a=chat_prompt(qa, [], p_ab, 1400, qa_style=qa_style),
            prompt_b=chat_prompt(qb, [], p_ab, 1400, qa_style=qa_style),
            answer_a=va, answer_b=vb,
        ),
        dict(
            id='q2', family=family, direction=direction, distractor=True,
            prompt_a=chat_prompt(qa, [], p_ba, 1400, qa_style=qa_style),
            prompt_b=chat_prompt(qb, [], p_ba, 1400, qa_style=qa_style),
            answer_a=vb, answer_b=va,
        ),
    ]


def generate_pairs(
    n_pairs: int,
    seed: int,
    vocab: dict[str, list[str]],
    *,
    families: tuple[str, ...] = ALL_FAMILIES,
    distractor_prob: float = 0.5,
    unseen_template: bool = False,
    filler_choices: tuple[int, ...] = (0,),
    nonce_value_prob: float = 0.0,
    nonce_key_prob: float = 0.0,
    qa_style: str = 'role',
    excluded_keys: set[tuple] | None = None,
) -> list[dict]:
    """Generate unique counterfactual value-swap pairs with optional exclusion set."""
    rng = random.Random(seed)
    seen = set(excluded_keys or ())
    pairs = []
    while len(pairs) < n_pairs:
        fam = families[len(pairs) % len(families)]
        dist = rng.random() < distractor_prob
        fillers = rng.choice(filler_choices)
        pair = _build_pair(
            rng, vocab, fam, dist, unseen_template, fillers, f'p{len(pairs):04d}',
            nonce_value_prob=nonce_value_prob, nonce_key_prob=nonce_key_prob, qa_style=qa_style,
        )
        key = (pair['prompt_a'], pair['prompt_b'])
        if key in seen:
            continue
        seen.add(key)
        pairs.append(pair)
    return pairs


def generate_qswap_pairs(
    n_pairs: int,
    seed: int,
    vocab: dict[str, list[str]],
    *,
    families: tuple[str, ...] = ('city', 'color', 'place', 'person'),
    filler_choices: tuple[int, ...] = (0,),
    nonce_val_prob: float = 0.0,
    nonce_key_prob: float = 0.0,
    qa_style: str = 'role',
) -> list[dict]:
    """Generate fixed held-out question-swap pairs (identical passage, swapped query)."""
    rng = random.Random(seed)
    pairs = []
    for i in range(n_pairs):
        fam = families[i % len(families)]
        fc = rng.choice(filler_choices)
        pair = make_quad_as_two_pairs(
            rng, vocab, fam, nonce_val_prob=nonce_val_prob, nonce_key_prob=nonce_key_prob,
            filler_count=fc, qa_style=qa_style,
        )[0]
        pair['id'] = f'qs{i:03d}'
        pairs.append(pair)
    return pairs


def stream_batch(
    rng: random.Random,
    n_pairs: int,
    vocab: dict[str, list[str]],
    *,
    families: tuple[str, ...] = ALL_FAMILIES,
    distractor_prob: float = 0.75,
    filler_count: int = 0,
    nonce_value_prob: float = 0.5,
    nonce_key_prob: float = 0.25,
    qa_style: str = 'role',
    excluded_keys: set[tuple] | None = None,
) -> list[dict]:
    """Draw a fresh online batch of value-swap pairs."""
    excluded = excluded_keys or set()
    batch = []
    while len(batch) < n_pairs:
        fam = rng.choice(families)
        dist = rng.random() < distractor_prob
        pair = _build_pair(
            rng, vocab, fam, dist, False, filler_count, f's{len(batch)}',
            nonce_value_prob=nonce_value_prob, nonce_key_prob=nonce_key_prob, qa_style=qa_style,
        )
        if (pair['prompt_a'], pair['prompt_b']) in excluded:
            continue
        batch.append(pair)
    return batch


def stream_quad_batch(
    rng: random.Random,
    n_quads: int,
    vocab: dict[str, list[str]],
    *,
    families: tuple[str, ...] = ('city', 'color', 'place', 'person'),
    clean_prob: float = 0.0,
    filler_choices: tuple[int, ...] = (0,),
    nonce_val_prob: float = 0.5,
    nonce_key_prob: float = 0.35,
    qa_style: str = 'role',
) -> list[dict]:
    """Draw n_quads (each = 2 pairs = 4 sequences): either a 2x2 question-swap quad or 2 clean pairs."""
    batch = []
    for _ in range(n_quads):
        fam = rng.choice(families)
        fc = rng.choice(filler_choices)
        if clean_prob > 0 and rng.random() < clean_prob:
            for k in range(2):
                batch.append(_build_pair(
                    rng, vocab, fam, False, False, fc, f'c{k}',
                    nonce_value_prob=nonce_val_prob, nonce_key_prob=0.0, qa_style=qa_style,
                ))
        else:
            batch.extend(make_quad_as_two_pairs(
                rng, vocab, fam, nonce_val_prob=nonce_val_prob, nonce_key_prob=nonce_key_prob,
                filler_count=fc, qa_style=qa_style,
            ))
    return batch


def first_diff_index(a: bytes, b: bytes) -> int:
    """First byte index where distinct byte strings a and b diverge."""
    for i, (ba, bb) in enumerate(zip(a, b)):
        if ba != bb:
            return i
    if len(a) != len(b):
        return min(len(a), len(b))
    raise ValueError('Answers must be distinct')


def pair_batch_tensors(pairs: list[dict], device: torch.device | str):
    """Pack N counterfactual pairs into 2N sequences [A0, B0, A1, B1, ...] for single-pass scoring/training."""
    seqs = []
    prompt_lens = []
    target_lens = []
    diff_info = []
    for pair in pairs:
        ta = (pair['answer_a'] + '\n').encode('utf8')
        tb = (pair['answer_b'] + '\n').encode('utf8')
        pa = pair['prompt_a'].encode('utf8')
        pb = pair['prompt_b'].encode('utf8')
        k = first_diff_index(ta, tb)
        row_a = len(seqs)
        seqs.append(pa + ta)
        prompt_lens.append(len(pa))
        target_lens.append(len(ta))
        diff_info.append((row_a, len(pa) - 1 + k, ta[k], tb[k]))
        row_b = len(seqs)
        seqs.append(pb + tb)
        prompt_lens.append(len(pb))
        target_lens.append(len(tb))
        diff_info.append((row_b, len(pb) - 1 + k, tb[k], ta[k]))

    max_len = max(len(s) - 1 for s in seqs)
    x = torch.zeros((len(seqs), max_len), dtype=torch.long)
    y = torch.full_like(x, -100)
    w = torch.zeros((len(seqs), max_len), dtype=torch.float32)
    for i, (seq, plen, tlen) in enumerate(zip(seqs, prompt_lens, target_lens)):
        toks = torch.tensor(list(seq), dtype=torch.long)
        end = len(seq) - 1
        start = plen - 1
        x[i, :end] = toks[:-1]
        y[i, start:end] = toks[start + 1:]
        w[i, start:end] = torch.arange(1, tlen + 1, dtype=torch.float32)
    diff_tensor = torch.tensor(diff_info, dtype=torch.long)
    return x.to(device), y.to(device), w.to(device), diff_tensor.to(device)


def fact_switch_loss(
    logits: torch.Tensor,
    y: torch.Tensor,
    byte_pos: torch.Tensor,
    diff_info: torch.Tensor,
    *,
    mode: str = 'contrastive',
    first_byte_weight: float = 2.0,
    margin_target: float = 3.0,
    contrastive_weight: float = 0.5,
) -> tuple[torch.Tensor, dict[str, float]]:
    """Compute plain NLL (Phase 1 copy lock-in) or counterfactual contrastive loss (Phase 2 routing)."""
    nll_tok = F.cross_entropy(logits.float().transpose(1, 2), y, ignore_index=-100, reduction='none')
    valid = y != -100
    plain_nll = nll_tok[valid].mean()

    rows, pos, true_b, foil_b = diff_info.unbind(1)
    div_logits = logits.float()[rows, pos]
    true_z = div_logits.gather(1, true_b[:, None]).squeeze(1)
    foil_z = div_logits.gather(1, foil_b[:, None]).squeeze(1)
    div_margin = true_z - foil_z
    pair_sens = div_margin.reshape(-1, 2).sum(1)

    if mode == 'plain_nll':
        loss = plain_nll
    elif mode == 'weighted_nll':
        weights = torch.where(byte_pos == 1, first_byte_weight, 1.0) * valid.float()
        loss = (nll_tok * weights).sum() / weights.sum().clamp_min(1.0)
    elif mode == 'contrastive':
        weights = torch.where(byte_pos == 1, first_byte_weight, 1.0) * valid.float()
        weighted_nll = (nll_tok * weights).sum() / weights.sum().clamp_min(1.0)
        indiv_hinge = F.softplus(margin_target - div_margin).mean()
        swap_hinge = F.softplus(2.0 * margin_target - pair_sens).mean()
        loss = weighted_nll + contrastive_weight * (0.5 * indiv_hinge + 0.5 * swap_hinge)
    else:
        raise ValueError(f'Unknown loss mode: {mode}')

    stats = dict(
        loss=float(loss.detach()),
        plain_nll=float(plain_nll.detach()),
        mean_div_margin=float(div_margin.detach().mean()),
        mean_pair_sens=float(pair_sens.detach().mean()),
        div_pair_acc=float((div_margin.detach().reshape(-1, 2) > 0).all(1).float().mean()),
    )
    return loss, stats


@torch.inference_mode()
def _score_prompt_endings(
    model: LM,
    prompts: list[str],
    endings: list[str],
    device: torch.device | str,
    mode: str = 'on',
    batch_size: int = 16,
) -> tuple[list[float], list[float], list[list[float]]]:
    model.eval()
    amp = torch.autocast('cuda', dtype=torch.bfloat16) if torch.device(device).type == 'cuda' else nullcontext()
    totals, per_bytes, details = [], [], []
    eff_bs = max(2, batch_size) if mode == 'shuffle' else batch_size
    for start in range(0, len(prompts), eff_bs):
        ps = prompts[start:start + eff_bs]
        es = endings[start:start + eff_bs]
        if mode == 'shuffle' and len(ps) == 1:
            ps = ps + [prompts[0]]
            es = es + [endings[0]]
            drop_last = True
        else:
            drop_last = False
        encoded = [(p.encode('utf8'), e.encode('utf8')) for p, e in zip(ps, es)]
        max_len = max(len(p) + len(e) - 1 for p, e in encoded)
        ids = torch.zeros((len(encoded), max_len), dtype=torch.long)
        for i, (pb, eb) in enumerate(encoded):
            seq = pb + eb
            ids[i, :len(seq) - 1] = torch.tensor(list(seq[:-1]), dtype=torch.long)
        ids = ids.to(device)
        with amp:
            logits = model(ids, mode=mode).float()
        logp = logits.log_softmax(-1)
        limit = len(encoded) - (1 if drop_last else 0)
        for i in range(limit):
            pb, eb = encoded[i]
            p0 = len(pb) - 1
            toks = torch.tensor(list(eb), device=device, dtype=torch.long)
            vals = logp[i, p0:p0 + len(eb)].gather(1, toks[:, None]).squeeze(1).tolist()
            totals.append(float(sum(vals)))
            per_bytes.append(float(sum(vals) / len(vals)))
            details.append(vals)
    return totals, per_bytes, details


@torch.inference_mode()
def greedy_generate(
    model: LM,
    prompts: list[str],
    device: torch.device | str,
    *,
    max_bytes: int = 20,
    mode: str = 'on',
    batch_size: int = 16,
) -> list[str]:
    """Batched greedy decode from each prompt up to newline or max_bytes."""
    if not prompts:
        return []
    model.eval()
    amp = torch.autocast('cuda', dtype=torch.bfloat16) if torch.device(device).type == 'cuda' else nullcontext()
    outputs = []
    for start in range(0, len(prompts), batch_size):
        chunk_prompts = prompts[start:start + batch_size]
        if mode == 'shuffle' and len(chunk_prompts) < 2:
            chunk_prompts = chunk_prompts + [prompts[0]]
            drop_last = True
        else:
            drop_last = False
        bufs = [list(p.encode('utf8')) for p in chunk_prompts]
        prompt_lens = [len(b) for b in bufs]
        done = [False] * len(bufs)
        for _ in range(max_bytes):
            if all(done):
                break
            max_len = max(len(b) for b in bufs)
            ids = torch.zeros((len(bufs), max_len), dtype=torch.long)
            last_pos = torch.tensor([len(b) - 1 for b in bufs], dtype=torch.long)
            for i, b in enumerate(bufs):
                ids[i, :len(b)] = torch.tensor(b, dtype=torch.long)
            ids = ids.to(device)
            with amp:
                logits = model(ids, mode=mode, positions=last_pos.to(device)[:, None]).squeeze(1).float()
            nxt = logits.argmax(-1).tolist()
            for i, byte in enumerate(nxt):
                if not done[i]:
                    bufs[i].append(int(byte))
                    if byte == 10:
                        done[i] = True
        limit = len(bufs) - (1 if drop_last else 0)
        for i in range(limit):
            gen = bytes(bufs[i][prompt_lens[i]:]).decode('utf8', errors='replace')
            outputs.append(gen.split('\n', 1)[0].strip())
    return outputs


def normalize_answer(text: str) -> str:
    return ' '.join(re.findall(r'\w+', text.casefold()))


@torch.inference_mode()
def evaluate_pairs(
    model: LM,
    pairs: list[dict],
    device: torch.device | str,
    *,
    mode: str = 'on',
    do_generate: bool = True,
    gen_limit: int | None = None,
    batch_size: int = 16,
) -> dict:
    """Evaluate forced-choice pair flipping, first-byte sensitivity, direction breakdown, and greedy generation."""
    prompts_a = [p['prompt_a'] for p in pairs]
    prompts_b = [p['prompt_b'] for p in pairs]
    ans_a = [p['answer_a'] + '\n' for p in pairs]
    ans_b = [p['answer_b'] + '\n' for p in pairs]

    s_aa, pb_aa, det_aa = _score_prompt_endings(model, prompts_a, ans_a, device, mode=mode, batch_size=batch_size)
    s_ab, pb_ab, det_ab = _score_prompt_endings(model, prompts_a, ans_b, device, mode=mode, batch_size=batch_size)
    s_ba, pb_ba, det_ba = _score_prompt_endings(model, prompts_b, ans_a, device, mode=mode, batch_size=batch_size)
    s_bb, pb_bb, det_bb = _score_prompt_endings(model, prompts_b, ans_b, device, mode=mode, batch_size=batch_size)

    gen_n = len(pairs) if gen_limit is None else min(len(pairs), gen_limit)
    gen_a = greedy_generate(model, prompts_a[:gen_n], device, mode=mode, batch_size=batch_size) if do_generate else None
    gen_b = greedy_generate(model, prompts_b[:gen_n], device, mode=mode, batch_size=batch_size) if do_generate else None

    rows = []
    for i, p in enumerate(pairs):
        ta = ans_a[i].encode('utf8')
        tb = ans_b[i].encode('utf8')
        k = first_diff_index(ta, tb)
        same_len = len(ta) == len(tb)
        dA = (s_aa[i] - s_ab[i]) if same_len else (pb_aa[i] - pb_ab[i])
        dB = (s_bb[i] - s_ba[i]) if same_len else (pb_bb[i] - pb_ba[i])
        fb_dA = det_aa[i][k] - det_ab[i][k]
        fb_dB = det_bb[i][k] - det_ba[i][k]
        row = dict(
            id=p['id'],
            family=p['family'],
            direction=p.get('direction', FAMILY_DIRECTION.get(p['family'], 'unknown')),
            distractor=p['distractor'],
            answer_a=p['answer_a'],
            answer_b=p['answer_b'],
            dA=dA,
            dB=dB,
            sens=dA + dB,
            abs_bias=abs(dA - dB) / 2.0,
            both_correct=(dA > 0) and (dB > 0),
            indiv_correct=int(dA > 0) + int(dB > 0),
            same_choice=((dA > 0) != (dB > 0)),
            fb_dA=fb_dA,
            fb_dB=fb_dB,
            fb_sens=fb_dA + fb_dB,
            fb_abs_bias=abs(fb_dA - fb_dB) / 2.0,
            fb_both_correct=(fb_dA > 0) and (fb_dB > 0),
        )
        if do_generate and i < gen_n:
            ga, gb = gen_a[i], gen_b[i]
            ea = normalize_answer(ga) == normalize_answer(p['answer_a'])
            eb = normalize_answer(gb) == normalize_answer(p['answer_b'])
            row.update(gen_a=ga, gen_b=gb, gen_a_exact=ea, gen_b_exact=eb, gen_both_exact=ea and eb)
        rows.append(row)

    n = len(rows)
    by_dir = {}
    grouped = defaultdict(list)
    for r in rows:
        grouped[r['direction']].append(r)
    for dname, rs in grouped.items():
        entry = dict(
            pairs=len(rs),
            pair_acc=sum(r['both_correct'] for r in rs) / len(rs),
            fb_mean_sens=float(np.mean([r['fb_sens'] for r in rs])),
        )
        grs = [r for r in rs if 'gen_both_exact' in r]
        if grs:
            entry['gen_pair_exact'] = sum(r['gen_both_exact'] for r in grs) / len(grs)
            entry['gen_indiv_exact'] = sum(int(r['gen_a_exact']) + int(r['gen_b_exact']) for r in grs) / (2 * len(grs))
        by_dir[dname] = entry

    summary = dict(
        pairs=n,
        mode=mode,
        pair_acc=sum(r['both_correct'] for r in rows) / n,
        indiv_acc=sum(r['indiv_correct'] for r in rows) / (2 * n),
        same_choice_rate=sum(r['same_choice'] for r in rows) / n,
        mean_sens=float(np.mean([r['sens'] for r in rows])),
        mean_abs_bias=float(np.mean([r['abs_bias'] for r in rows])),
        fb_pair_acc=sum(r['fb_both_correct'] for r in rows) / n,
        fb_mean_sens=float(np.mean([r['fb_sens'] for r in rows])),
        fb_mean_abs_bias=float(np.mean([r['fb_abs_bias'] for r in rows])),
        by_direction=by_dir,
        rows=rows,
    )
    if do_generate and gen_n > 0:
        gen_rows = rows[:gen_n]
        summary['gen_pairs_evaluated'] = gen_n
        summary['gen_indiv_exact'] = sum(int(r['gen_a_exact']) + int(r['gen_b_exact']) for r in gen_rows) / (2 * gen_n)
        summary['gen_pair_exact'] = sum(r['gen_both_exact'] for r in gen_rows) / gen_n
    return summary


@torch.inference_mode()
def evaluate_tree_counterfactual_swap(model: LM, pairs: list[dict], device: torch.device | str) -> dict:
    """Directly swap Tree source activations between prompt_a and prompt_b while keeping Delta & Attn intact."""
    model.eval()
    amp = torch.autocast('cuda', dtype=torch.bfloat16) if torch.device(device).type == 'cuda' else nullcontext()
    deltas_on, deltas_swap = [], []
    for p in pairs:
        ta = (p['answer_a'] + '\n').encode('utf8')
        tb = (p['answer_b'] + '\n').encode('utf8')
        k = first_diff_index(ta, tb)
        pa = p['prompt_a'].encode('utf8')
        pb = p['prompt_b'].encode('utf8')
        seqs = [pa + ta, pb + tb]
        max_len = max(len(s) - 1 for s in seqs)
        ids = torch.zeros((2, max_len), dtype=torch.long)
        for i, s in enumerate(seqs):
            ids[i, :len(s) - 1] = torch.tensor(list(s[:-1]), dtype=torch.long)
        ids = ids.to(device)
        pos_a, pos_b = len(pa) - 1 + k, len(pb) - 1 + k
        with amp:
            lon = model(ids, mode='on').float()
            lsw = model(ids, mode='shuffle').float()
        sens_on = float((lon[0, pos_a, ta[k]] - lon[0, pos_a, tb[k]]) + (lon[1, pos_b, tb[k]] - lon[1, pos_b, ta[k]]))
        sens_sw = float((lsw[0, pos_a, ta[k]] - lsw[0, pos_a, tb[k]]) + (lsw[1, pos_b, tb[k]] - lsw[1, pos_b, ta[k]]))
        deltas_on.append(sens_on)
        deltas_swap.append(sens_sw)
    return dict(
        pairs=len(pairs),
        fb_sens_on=float(np.mean(deltas_on)),
        fb_sens_tree_swapped=float(np.mean(deltas_swap)),
        tree_swap_drop=float(np.mean(deltas_on) - np.mean(deltas_swap)),
    )


@torch.inference_mode()
def evaluate_coqa_extractive(
    model: LM,
    coqa_rows: list[tuple[str, str]],
    device: torch.device | str,
    mode: str = 'on',
) -> dict:
    """Evaluate greedy exact match & token F1 on real held-out CoQA extractive questions."""
    prompts = [p for p, _ in coqa_rows]
    refs = [t.strip() for _, t in coqa_rows]
    preds = greedy_generate(model, prompts, device, max_bytes=28, mode=mode, batch_size=8)
    exacts, f1s, samples = [], [], []
    for ref, pred in zip(refs, preds):
        nr, np_ = normalize_answer(ref), normalize_answer(pred)
        exact = nr == np_
        rt, pt = nr.split(), np_.split()
        if not rt or not pt:
            f1 = float(rt == pt)
        else:
            common = 0
            rem = list(rt)
            for tok in pt:
                if tok in rem:
                    common += 1
                    rem.remove(tok)
            f1 = (2 * common / (len(rt) + len(pt))) if common else 0.0
        exacts.append(exact)
        f1s.append(f1)
        samples.append(dict(reference=ref, predicted=pred, exact=exact, f1=f1))
    return dict(
        examples=len(coqa_rows),
        exact_match=float(np.mean(exacts)),
        token_f1=float(np.mean(f1s)),
        samples=samples[:8],
    )


def build_fact_switch_suite(qa_style: str = 'role', pairs_per_split: int = 16) -> dict[str, list[dict]]:
    """Standard OOV fact-switch diagnostic suite separating Phase-1 copying from Phase-2 key binding."""
    return {
        'user_6pairs': user_audit_pairs(qa_style=qa_style),
        'user_qswap': user_qswap_pairs(qa_style=qa_style),
        'oov_clean': generate_pairs(
            pairs_per_split, seed=2001, vocab=OOV_VOCAB,
            families=('city', 'color', 'place', 'person'), distractor_prob=0.0, qa_style=qa_style,
        ),
        'oov_qswap': generate_qswap_pairs(
            pairs_per_split, seed=2002, vocab=OOV_VOCAB,
            families=('city', 'color', 'place', 'person'), qa_style=qa_style,
        ),
        'nonce_qswap': generate_qswap_pairs(
            pairs_per_split, seed=2005, vocab=OOV_VOCAB,
            families=('city', 'color', 'place', 'person'), nonce_val_prob=1.0, nonce_key_prob=1.0, qa_style=qa_style,
        ),
        'long_clean': generate_pairs(
            pairs_per_split, seed=2004, vocab=OOV_VOCAB,
            families=('city', 'color', 'place', 'person'), distractor_prob=0.0,
            filler_choices=(2,), qa_style=qa_style,
        ),
    }


def evaluate_fact_switch_suite(
    model: LM,
    suite: dict[str, list[dict]],
    device: torch.device | str,
    *,
    do_generate: bool = True,
    gen_limit: int = 8,
    run_tree_ablations: bool = False,
) -> dict:
    """Evaluate all fact-switch splits plus optional causal Tree ablations."""
    results = {}
    for name, items in suite.items():
        limit = len(items) if name in ('user_6pairs', 'user_qswap') else gen_limit
        r = evaluate_pairs(model, items, device, mode='on', do_generate=do_generate, gen_limit=limit)
        if name in ('user_6pairs', 'user_qswap'):
            results[name] = r
        else:
            compact = {k: v for k, v in r.items() if k != 'rows'}
            if do_generate:
                compact['sample_generations'] = [
                    {k: row[k] for k in ('id', 'family', 'answer_a', 'answer_b', 'gen_a', 'gen_b', 'gen_both_exact') if k in row}
                    for row in r['rows'][:4]
                ]
            results[name] = compact
    if run_tree_ablations and model.tree is not None:
        ablations = {}
        for sp in ('long_clean', 'oov_qswap'):
            if sp not in suite:
                continue
            items = suite[sp]
            abl = {
                m: {k: v for k, v in evaluate_pairs(model, items, device, mode=m, do_generate=False).items() if k != 'rows'}
                for m in ('on', 'off', 'shuffle')
            }
            abl['counterfactual_tree_swap'] = evaluate_tree_counterfactual_swap(model, items, device)
            ablations[sp] = abl
        results['tree_ablations'] = ablations
    return results


def fact_switch_selection_key(summary: dict) -> tuple[float, float, float, float]:
    """Rank checkpoints by Phase-2 key-matched qswap first, then clean OOV & greedy generation."""
    qswap = summary.get('oov_qswap', {})
    clean = summary.get('oov_clean', {})
    user_q = summary.get('user_qswap', {})
    qswap_pair = 0.5 * qswap.get('pair_acc', 0.0) + 0.5 * user_q.get('pair_acc', 0.0)
    clean_pair = clean.get('pair_acc', 0.0)
    gen_exact = 0.5 * qswap.get('gen_indiv_exact', 0.0) + 0.5 * clean.get('gen_indiv_exact', 0.0)
    sens = qswap.get('fb_mean_sens', 0.0) + 0.25 * clean.get('fb_mean_sens', 0.0)
    return (round(qswap_pair, 4), round(gen_exact, 4), round(clean_pair, 4), round(sens, 4))


def format_fact_switch_table(label: str, summary: dict) -> str:
    lines = [f'=== {label} ===']
    for sp in ('user_6pairs', 'user_qswap', 'oov_clean', 'oov_qswap', 'nonce_qswap', 'long_clean'):
        if sp not in summary:
            continue
        v = summary[sp]
        gen_s = f" | gen_pair={v['gen_pair_exact']:.2f} gen_ind={v['gen_indiv_exact']:.2f}" if 'gen_pair_exact' in v else ''
        lines.append(
            f"  {sp:12s}: pair={v['pair_acc']:.2f} fb_pair={v['fb_pair_acc']:.2f} "
            f"sens={v['mean_sens']:+5.2f} fb_sens={v['fb_mean_sens']:+5.2f}{gen_s}"
        )
    if 'tree_ablations' in summary:
        lines.append('  Causal Tree ablations (fb_sens):')
        for sp, abl in summary['tree_ablations'].items():
            lines.append(
                f"    {sp:10s}: on={abl['on']['fb_mean_sens']:+5.2f} "
                f"off={abl['off']['fb_mean_sens']:+5.2f} "
                f"shuf={abl['shuffle']['fb_mean_sens']:+5.2f} "
                f"cf_swap={abl['counterfactual_tree_swap']['fb_sens_tree_swapped']:+5.2f} "
                f"(drop={abl['counterfactual_tree_swap']['tree_swap_drop']:+5.2f})"
            )
    return '\n'.join(lines)


def load_coqa_pools(dialogue_dir: str | Path, context: int = 1024, eval_limit: int = 16, seed: int = 42, qa_style: str = 'role'):
    """Load training qa_excerpt rows and held-out extractive CoQA dev questions."""
    data, _ = read_dialogue_data(dialogue_dir, offline=True)
    pools = prepare_dialogues(data, context, qa_style=qa_style)
    train_excerpts = [(p, t) for p, t in pools['train']['qa_excerpt'] if len((p + t).encode()) <= 420]
    eligible = []
    for prompt, target in pools['heldout']['qa']:
        ans = single_line(target).casefold()
        if ans and ans not in ('yes', 'no', 'unknown') and len(ans) <= 24 and ans in prompt.casefold():
            eligible.append((prompt, target))
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(eligible), min(eval_limit, len(eligible)), replace=False)
    heldout = [eligible[int(i)] for i in idx]
    return train_excerpts, heldout


def run_two_stage_fact_switch(
    model: LM,
    suite: dict[str, list[dict]],
    device: torch.device | str,
    *,
    minutes: float = 6.0,
    max_steps: int | None = None,
    quads_per_step: int = 2,
    lr: float = 5e-4,
    qa_style: str = 'qa',
    seed: int = 42,
    replay_fn=None,
    replay_every: int = 0,
    eval_interval_seconds: float = 60.0,
    event_fn=None,
) -> tuple[LM, dict]:
    """Two-stage fact-to-answer adaptation:
      Stage 1 (first 35% budget): plain_nll on mixed clean + 2x2 quads + 60% nonce values (copy-head lock-in).
      Stage 2 (remaining 65%):    contrastive loss on 100% 2x2 counterfactual quads + nonce keys (Phase-2 routing).
    Selects the best weights by counterfactual pair accuracy + greedy exact match (never BPB alone).
    """
    dev = torch.device(device)
    amp = torch.autocast('cuda', dtype=torch.bfloat16) if dev.type == 'cuda' else nullcontext()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01, fused=(dev.type == 'cuda'))
    py_rng = random.Random(seed + 401)

    t0 = time.perf_counter()
    deadline = t0 + 60.0 * minutes if minutes > 0 else float('inf')
    stage1_deadline = t0 + 60.0 * minutes * 0.35 if minutes > 0 else float('inf')

    before = evaluate_fact_switch_suite(model, suite, dev, do_generate=True, run_tree_ablations=True)
    best_key = fact_switch_selection_key(before)
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_step, best_summary = 0, before
    if event_fn:
        event_fn(stage='fact_switch_initial', key=list(best_key),
                 oov_clean=before['oov_clean']['pair_acc'], oov_qswap=before['oov_qswap']['pair_acc'])

    step, last_check = 0, time.perf_counter()
    last_stats = {}
    while (max_steps is None or step < max_steps) and time.perf_counter() < deadline:
        in_stage1 = (
            (step < int(0.35 * max_steps)) if max_steps is not None
            else (time.perf_counter() < stage1_deadline)
        )
        warmup = 20
        cur_lr = lr * min(1.0, (step + 1) / warmup) * (1.0 if in_stage1 else 0.7)
        for grp in opt.param_groups:
            grp['lr'] = cur_lr

        model.train()
        opt.zero_grad(set_to_none=True)
        if replay_fn is not None and replay_every > 0 and (step + 1) % replay_every == 0:
            x, y = replay_fn()
            with amp:
                loss = F.cross_entropy(model(x).float().flatten(0, 1), y.reshape(-1))
        else:
            batch = stream_quad_batch(
                py_rng, quads_per_step, TRAIN_VOCAB,
                clean_prob=0.35 if in_stage1 else 0.0,
                filler_choices=(0,) if in_stage1 else (0, 0, 1),
                nonce_val_prob=0.6 if in_stage1 else 0.5,
                nonce_key_prob=0.25 if in_stage1 else 0.5,
                qa_style=qa_style,
            )
            x, y, bpos, dinfo = pair_batch_tensors(batch, dev)
            with amp:
                logits = model(x)
                loss, last_stats = fact_switch_loss(
                    logits, y, bpos, dinfo,
                    mode='plain_nll' if in_stage1 else 'contrastive',
                    first_byte_weight=2.5, margin_target=3.0, contrastive_weight=0.75,
                )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
        opt.step()
        step += 1

        now = time.perf_counter()
        if now - last_check >= eval_interval_seconds or (max_steps is not None and step == max_steps) or now >= deadline:
            val_sum = evaluate_fact_switch_suite(model, suite, dev, do_generate=True, gen_limit=6, run_tree_ablations=False)
            key = fact_switch_selection_key(val_sum)
            if key > best_key:
                best_key, best_step, best_summary = key, step, val_sum
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if event_fn:
                event_fn(
                    stage='fact_switch_val', step=step, phase=1 if in_stage1 else 2,
                    oov_clean=val_sum['oov_clean']['pair_acc'], oov_qswap=val_sum['oov_qswap']['pair_acc'],
                    user6=val_sum['user_6pairs']['pair_acc'], user_qswap=val_sum['user_qswap']['pair_acc'],
                    gen_oov_clean=val_sum['oov_clean'].get('gen_indiv_exact', 0.0),
                    selected_step=best_step,
                )
            last_check = time.perf_counter()

    model.load_state_dict(best_state)
    model.eval()
    selected = evaluate_fact_switch_suite(model, suite, dev, do_generate=True, run_tree_ablations=True)
    return model, dict(
        steps=step,
        selected_step=best_step,
        quads_per_step=quads_per_step,
        lr=lr,
        qa_style=qa_style,
        seconds=time.perf_counter() - t0,
        last_train_stats=last_stats,
        before=before,
        selected=selected,
    )


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--checkpoint', default='runs/best_lm.pt')
    p.add_argument('--out', default='')
    p.add_argument('--device', default='cpu', choices=('cpu', 'cuda'))
    p.add_argument('--backend', default='torch', choices=('torch', 'fla'))
    p.add_argument('--minutes', type=float, default=5.0)
    p.add_argument('--max-steps', type=int, default=None)
    p.add_argument('--quads-per-step', type=int, default=2)
    p.add_argument('--lr', type=float, default=5e-4)
    p.add_argument('--qa-style', default='qa', choices=('role', 'qa'))
    p.add_argument('--read-window', type=int, default=1536)
    p.add_argument('--widen-l3', action='store_true')
    a = p.parse_args()

    device = torch.device(a.device)
    ck = torch.load(a.checkpoint, map_location='cpu', weights_only=True)
    out_dir = Path(a.out) if a.out else Path('runs') / time.strftime('fact-switch-%Y%m%d-%H%M%S')
    out_dir.mkdir(parents=True, exist_ok=True)
    model = LM(
        ck['kind'], ck['slots'], backend=a.backend,
        read_window=a.read_window if ck['kind'] == 'C1' else None,
        mid_window=a.read_window if (ck['kind'] == 'C1' and a.widen_l3) else None,
    ).to(device)
    model.load_state_dict(ck['state_dict'])
    suite = build_fact_switch_suite(qa_style=a.qa_style, pairs_per_split=16)
    model, res = run_two_stage_fact_switch(
        model, suite, device,
        minutes=a.minutes, max_steps=a.max_steps, quads_per_step=a.quads_per_step,
        lr=a.lr, qa_style=a.qa_style, event_fn=lambda **row: print(row, flush=True),
    )
    print(format_fact_switch_table('Before', res['before']))
    print(format_fact_switch_table('Selected', res['selected']))
    torch.save(dict(
        kind=ck['kind'], slots=ck['slots'],
        read_window=model.blocks[-1].mix.window if model.kind == 'C1' else None,
        mid_window=model.blocks[3].mix.window if model.kind == 'C1' else None,
        state_dict=model.state_dict(), qa_style=a.qa_style,
    ), out_dir / 'best_fact_switch.pt')
    (out_dir / 'report.json').write_text(json.dumps(res, indent=2), encoding='utf8')
    print(f'Saved results to {out_dir / "report.json"}')


if __name__ == '__main__' and '__file__' in globals():
    main()

In [ ]:
#@title Priority #1: Counterfactual fact-to-answer stage and optional A100 showdown
fact_suite = build_fact_switch_suite(qa_style=QA_PROMPT_STYLE, pairs_per_split=FACT_SWITCH_EVAL_PAIRS)
fact_base_eval = evaluate_fact_switch_suite(prose_model, fact_suite, device, do_generate=True, run_tree_ablations=True)
print(format_fact_switch_table('Zero-shot prose checkpoint', fact_base_eval), flush=True)

fact_switch_result = dict(
    qa_style=QA_PROMPT_STYLE,
    minutes_requested=FACT_SWITCH_MINUTES,
    widen_l3=DIALOGUE_WIDEN_L3,
    zero_shot=fact_base_eval,
    checkpoint=None,
)
fact_model = prose_model

def auto_quads_per_step(net):
    if DEVICE != 'cuda':
        return 2
    if not AUTO_BATCH:
        return max(1, BATCH_BYTES // (4 * 256))
    probe_rng = random.Random(SEED + 909)
    limit = allocation_limit()
    def measure(n_quads):
        trial = deepcopy(net).train()
        opt = torch.optim.AdamW(trial.parameters(), lr=FACT_SWITCH_LR, weight_decay=.01, fused=True)
        def step_once():
            batch = stream_quad_batch(probe_rng, n_quads, TRAIN_VOCAB, filler_choices=(1,), qa_style=QA_PROMPT_STYLE)
            x, y, bpos, dinfo = pair_batch_tensors(batch, device)
            opt.zero_grad(set_to_none=True)
            with precision(device):
                loss, _ = fact_switch_loss(trial(x), y, bpos, dinfo, mode='contrastive')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trial.parameters(), 1., error_if_nonfinite=True)
            opt.step()
        try:
            seconds, peak = timed_updates(step_once)
        finally:
            del trial, opt
        return dict(batch=n_quads, quads_per_second=n_quads/seconds,
                    median_train_step_seconds=seconds, peak_allocated_bytes=peak)
    max_quads = max(2, MAX_BATCH_BYTES // (4 * 256))
    best, _ = choose_batch(measure, 4, 1, max_quads, limit, 'quads_per_second')
    return best['batch']

if FACT_SWITCH_MINUTES > 0:
    fs_net = deepcopy(prose_model)
    fs_net.blocks[-1].mix.window = DIALOGUE_CONTEXT
    if DIALOGUE_WIDEN_L3:
        fs_net.blocks[3].mix.window = DIALOGUE_CONTEXT
    fs_quads = auto_quads_per_step(fs_net)
    print(f'Fact-switch stage: {fs_quads} quads/step ({4*fs_quads} sequences), L3={fs_net.blocks[3].mix.window}, L7={fs_net.blocks[-1].mix.window}', flush=True)
    fs_wiki_rng = np.random.default_rng(SEED + 409)
    wiki_replay_n = max(2, min(batch_size if TRAIN_MINUTES else 8, BATCH_BYTES // CONTEXT))
    fs_net, fs_run = run_two_stage_fact_switch(
        fs_net, fact_suite, device,
        minutes=FACT_SWITCH_MINUTES,
        max_steps=MAX_UPDATES,
        quads_per_step=fs_quads,
        lr=FACT_SWITCH_LR,
        qa_style=QA_PROMPT_STYLE,
        seed=SEED,
        replay_fn=lambda: train_batch(fs_wiki_rng, CONTEXT, wiki_replay_n),
        replay_every=6,
        eval_interval_seconds=60.0,
        event_fn=event,
    )
    fs_path = run_dir/'best_fact_switch.pt'
    tmp = fs_path.with_suffix('.partial')
    torch.save(dict(
        kind='C1', slots=8,
        read_window=fs_net.blocks[-1].mix.window,
        mid_window=fs_net.blocks[3].mix.window,
        qa_style=QA_PROMPT_STYLE,
        state_dict=fs_net.state_dict(),
        base_checkpoint=str(run_dir/'best_lm.pt'),
        step=best_step, bytes_seen=selected_prose_bytes,
        fact_switch_step=fs_run['selected_step'],
    ), tmp)
    tmp.replace(fs_path)
    fact_model = fs_net.eval()
    fact_switch_result.update(fs_run, checkpoint=str(fs_path))
    print(format_fact_switch_table(f"Selected fact-switch checkpoint (step {fs_run['selected_step']}/{fs_run['steps']})", fs_run['selected']), flush=True)
    release_memory()

save_json('fact_switch_results.json', fact_switch_result)

showdown_result = None
if RUN_ARCHITECTURE_SHOWDOWN:
    sd_quads = 8 if DEVICE == 'cuda' else 1
    sd_arms = [
        ('c1_l7',   dict(kind='C1', slots=8, from_base=True,  rw=DIALOGUE_CONTEXT, mw=64,               lr=FACT_SWITCH_LR)),
        ('c1_l3l7', dict(kind='C1', slots=8, from_base=True,  rw=DIALOGUE_CONTEXT, mw=DIALOGUE_CONTEXT, lr=FACT_SWITCH_LR)),
        ('c1_rw64', dict(kind='C1', slots=8, from_base=True,  rw=64,               mw=64,               lr=FACT_SWITCH_LR)),
        ('a0_10m',  dict(kind='A0', slots=0, from_base=False, rw=None,             mw=None,             lr=0.001)),
    ]
    showdown_result = dict(steps=SHOWDOWN_STEPS, quads_per_step=sd_quads, qa_style=QA_PROMPT_STYLE, arms={})
    base_sd = {k: v.detach().cpu().clone() for k, v in prose_model.state_dict().items()}
    for arm_name, spec in sd_arms:
        torch.manual_seed(SEED)
        net = LM(spec['kind'], spec['slots'], backend=backend,
                 read_window=spec['rw'] if spec['kind'] == 'C1' else None,
                 mid_window=spec['mw'] if spec['kind'] == 'C1' else None).to(device)
        if spec['from_base']:
            net.load_state_dict(base_sd)
        net, run_out = run_two_stage_fact_switch(
            net, fact_suite, device,
            minutes=0.0, max_steps=SHOWDOWN_STEPS, quads_per_step=sd_quads,
            lr=spec['lr'], qa_style=QA_PROMPT_STYLE, seed=SEED,
            eval_interval_seconds=90.0, event_fn=event,
        )
        print(format_fact_switch_table(f'Showdown {arm_name}', run_out['selected']), flush=True)
        showdown_result['arms'][arm_name] = run_out
        del net
        release_memory()
    save_json('fact_switch_showdown.json', showdown_result)

In [ ]:
#@title Priority #2: Rebalanced QA, sustained prose & conversation (A100 auto-batched)
def fixed_dialogue_rows(pools, count, seed):
    rng = np.random.default_rng(seed)
    result = {}
    for task, rows in pools.items():
        if not rows:
            raise ValueError(f'No eligible {task} examples. Check the data and DIALOGUE_CONTEXT.')
        result[task] = [rows[i] for i in rng.choice(len(rows), min(count, len(rows)), replace=False)]
    return result

@torch.inference_mode()
def score_dialogues(net, examples):
    net.eval()
    scores = {}
    for task, rows in examples.items():
        total, count = 0., 0
        for offset in range(0, len(rows), 2):
            x, y = target_batch(rows[offset:offset+2], device)
            with precision(device):
                loss = F.cross_entropy(net(x).float().flatten(0,1), y.flatten(), reduction='sum')
            total += float(loss)
            count += int((y != -100).sum())
        scores[task] = total / (count * math.log(2))
    return scores

def dialogue_selection_score(bpb_metrics, cf_summary):
    mean_bpb = sum(bpb_metrics.values()) / len(bpb_metrics)
    qswap_acc = cf_summary['oov_qswap']['pair_acc']
    clean_acc = cf_summary['oov_clean']['pair_acc']
    # Lower is better: preserve counterfactual fact-switching alongside held-out BPB.
    return mean_bpb - 0.35 * qswap_acc - 0.15 * clean_acc

def tune_dialogue_batch(net, sample_rows):
    # Fix: never clamp CUDA dialogue batch size to prose_tuning['selected_batch']=2 when TRAIN_MINUTES==0.
    base_size = max(1, BATCH_BYTES // DIALOGUE_CONTEXT)
    if DEVICE != 'cuda' or not AUTO_BATCH:
        return max(1, min(base_size, 2 if DEVICE == 'cpu' else base_size))
    limit = allocation_limit()
    longest = sorted(sample_rows, key=lambda r: len((r[0] + r[1]).encode()), reverse=True)
    def measure(n):
        trial = deepcopy(net).train()
        opt = torch.optim.AdamW(trial.parameters(), lr=DIALOGUE_LR, weight_decay=.01, fused=True)
        rows = [longest[i % len(longest)] for i in range(n)]
        def one_step():
            x, y = target_batch(rows, device, history_targets=True)
            opt.zero_grad(set_to_none=True)
            with precision(device):
                loss = F.cross_entropy(trial(x).float().flatten(0,1), y.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trial.parameters(), 1., error_if_nonfinite=True)
            opt.step()
        try:
            seconds, peak = timed_updates(one_step)
        finally:
            del trial, opt
        return dict(batch=n, seqs_per_second=n/seconds,
                    median_train_step_seconds=seconds, peak_allocated_bytes=peak)
    maximum = max(2, MAX_BATCH_BYTES // DIALOGUE_CONTEXT)
    best, _ = choose_batch(measure, max(2, base_size), 1, maximum, limit, 'seqs_per_second')
    return best['batch']

def train_dialogues(base):
    source_dir = Path(LOCAL_DIALOGUE_DIR) if LOCAL_DIALOGUE_DIR else cache/'conversation_data'
    records, hashes = read_dialogue_data(source_dir, offline=bool(LOCAL_DIALOGUE_DIR))
    pools = prepare_dialogues(records, DIALOGUE_CONTEXT, qa_style=QA_PROMPT_STYLE)
    del records
    counts = {split: {task: len(rows) for task, rows in group.items()} for split, group in pools.items()}
    print('Eligible examples:', counts, flush=True)
    valid = fixed_dialogue_rows(pools['validation'], DIALOGUE_EVAL_ROWS, SEED+71)
    heldout = fixed_dialogue_rows(pools['heldout'], DIALOGUE_EVAL_ROWS, SEED+72)
    if not all(counts['train'][task] for task in valid):
        raise ValueError('Need training examples for all three tasks.')

    val_cf_suite = dict(
        oov_clean=fact_suite['oov_clean'][:8],
        oov_qswap=fact_suite['oov_qswap'][:8],
        user_qswap=fact_suite['user_qswap'],
    )
    baseline = score_dialogues(base, valid)
    baseline['wiki'] = evaluate(base, 'validation')
    base_cf = evaluate_fact_switch_suite(base, val_cf_suite, device, do_generate=False)
    best_score, best_update = dialogue_selection_score(baseline, base_cf), 0
    best_metrics = dict(baseline)
    best_cf = {k: dict(pair=v['pair_acc'], fb_sens=v['fb_mean_sens']) for k, v in base_cf.items()}
    path = run_dir/'best_dialogue.pt'
    def save_candidate(net):
        tmp = path.with_suffix('.partial')
        torch.save(dict(
            kind='C1', slots=8,
            read_window=net.blocks[-1].mix.window,
            mid_window=net.blocks[3].mix.window,
            qa_style=QA_PROMPT_STYLE,
            state_dict=net.state_dict(),
            base_checkpoint=fact_switch_result.get('checkpoint') or str(run_dir/'best_lm.pt'),
            step=best_step, bytes_seen=selected_prose_bytes,
            dialogue_update=best_update, validation_bpb=best_metrics, validation_fact_switch=best_cf,
            context=DIALOGUE_CONTEXT, prompt_format=f'role-lines-{QA_PROMPT_STYLE}-v2',
        ), tmp)
        tmp.replace(path)
    save_candidate(base)

    net = deepcopy(base)
    net.blocks[-1].mix.window = DIALOGUE_CONTEXT
    if DIALOGUE_WIDEN_L3:
        net.blocks[3].mix.window = DIALOGUE_CONTEXT
    opt = torch.optim.AdamW(net.parameters(), lr=DIALOGUE_LR, weight_decay=.01, fused=(DEVICE=='cuda'))
    rng = np.random.default_rng(SEED+73)
    py_rng = random.Random(SEED+74)

    size = tune_dialogue_batch(net, pools['train']['qa'][:16] or pools['train']['chat'][:16])
    short_qa = 'qa_excerpt' if pools['train']['qa_excerpt'] else 'qa'
    ext_qa = 'qa_extractive' if pools['train']['qa_extractive'] else 'qa'
    long_prose = 'prose_long' if pools['train']['prose_long'] else 'prose'
    # Rebalanced 10-step cycle: grounded QA + 2x2 counterfactual quads get 40% of updates
    # (instead of ~1.6% of target bytes), plus multi-sentence sustained prose and WikiText anchors.
    schedule = (short_qa, 'fact_quad', ext_qa, long_prose, 'chat', 'fact_quad', 'qa', 'prose', 'chat', 'wiki')
    updates, supervised = 0, 0
    print(f'Dialogue stage: base batch={size} (L3={net.blocks[3].mix.window}, L7={net.blocks[-1].mix.window}, qa_style={QA_PROMPT_STYLE})', flush=True)

    def backward(n):
        task = schedule[updates % len(schedule)]
        net.train()
        opt.zero_grad(set_to_none=True)
        if task == 'fact_quad':
            quad_rng = random.Random(int(rng.integers(2**32)))
            n_quads = max(1, n // 2 if DEVICE == 'cuda' else min(2, n))
            batch = stream_quad_batch(
                quad_rng, n_quads, TRAIN_VOCAB,
                clean_prob=0.2, filler_choices=(0, 0, 1),
                nonce_val_prob=0.5, nonce_key_prob=0.4, qa_style=QA_PROMPT_STYLE,
            )
            x, y, bpos, dinfo = pair_batch_tensors(batch, device)
            with precision(device):
                loss, _ = fact_switch_loss(
                    net(x), y, bpos, dinfo,
                    mode='contrastive', first_byte_weight=2.5, margin_target=3.0, contrastive_weight=0.6,
                )
        elif task == 'wiki':
            wiki_n = max(1, min(n * 2, MAX_BATCH_BYTES // CONTEXT if dynamic else max(2, BATCH_BYTES // CONTEXT)))
            x, y = train_batch(rng, CONTEXT, wiki_n)
            with precision(device):
                loss = F.cross_entropy(net(x).float().flatten(0,1), y.reshape(-1))
        else:
            rows = pools['train'][task]
            eff_n = min(len(rows), n * 2) if (dynamic and task in ('qa_excerpt', 'prose')) else n
            selected_rows = [rows[i] for i in rng.integers(len(rows), size=eff_n)]
            x, y = target_batch(selected_rows, device, history_targets=task in ('qa', 'qa_extractive', 'chat'))
            with precision(device):
                loss = F.cross_entropy(net(x).float().flatten(0,1), y.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1., error_if_nonfinite=True)
        return loss.detach(), int((y != -100).sum())

    def validate_candidate():
        nonlocal best_score, best_update, best_metrics, best_cf
        metrics = score_dialogues(net, valid)
        metrics['wiki'] = evaluate(net, 'validation')
        cf = evaluate_fact_switch_suite(net, val_cf_suite, device, do_generate=False)
        value = dialogue_selection_score(metrics, cf)
        if not math.isfinite(value):
            raise FloatingPointError('Nonfinite dialogue validation loss')
        cf_compact = {k: dict(pair=v['pair_acc'], fb_sens=v['fb_mean_sens']) for k, v in cf.items()}
        if value < best_score:
            best_score, best_update, best_metrics, best_cf = value, updates, metrics, cf_compact
            save_candidate(net)
        event(stage='dialogue_validation', update=updates, bpb=metrics, fact_switch=cf_compact,
              composite=round(value, 4), selected_update=best_update)

    event(stage='dialogue_initial', bpb=baseline, fact_switch=best_cf)
    start = last_check = time.perf_counter()
    checked = 0
    while time.perf_counter()-start < 60*DIALOGUE_MINUTES and (MAX_UPDATES is None or updates < MAX_UPDATES):
        for group in opt.param_groups:
            group['lr'] = DIALOGUE_LR * min(1., (updates+1)/20)
        (loss, ntarget), size = backward_with_backoff(backward, opt, rng, size, 1, dynamic)
        opt.step()
        updates += 1
        supervised += ntarget
        if updates == 1 or updates % 100 == 0:
            event(stage='dialogue_train', update=updates, loss=float(loss), supervised_bytes=supervised, batch=size)
        if time.perf_counter()-last_check >= 90:
            validate_candidate()
            last_check, checked = time.perf_counter(), updates
    if updates != checked:
        validate_candidate()
    if updates == 0:
        raise RuntimeError('The dialogue stage was enabled but completed zero optimizer updates.')
    del opt, net
    release_memory()
    result = dict(
        checkpoint=str(path),
        base_checkpoint=fact_switch_result.get('checkpoint') or str(run_dir/'best_lm.pt'),
        counts=counts, sources=SOURCES, sha256=hashes,
        updates=updates, selected_update=best_update,
        validation_before=baseline, validation_selected=best_metrics,
        validation_fact_switch_selected=best_cf,
        supervised_bytes=supervised, context_bytes=DIALOGUE_CONTEXT,
        qa_style=QA_PROMPT_STYLE, widen_l3=DIALOGUE_WIDEN_L3,
        minutes_requested=DIALOGUE_MINUTES, eval_rows_per_task=DIALOGUE_EVAL_ROWS,
        train_seed=SEED+73, validation_seed=SEED+71, heldout_seed=SEED+72,
        lr=DIALOGUE_LR, train_seconds=time.perf_counter()-start, final_batch=size,
        schedule=list(schedule),
        selection='mean validation BPB - 0.35*oov_qswap_pair - 0.15*oov_clean_pair',
    )
    return result, heldout

dialogue_result, dialogue_heldout = None, None
if DIALOGUE_MINUTES > 0:
    dialogue_result, dialogue_heldout = train_dialogues(fact_model)
    save_json('dialogue_results.json', dialogue_result)
    print('Selected dialogue update', dialogue_result['selected_update'], 'of', dialogue_result['updates'])

## Read the saved checkpoint’s answers

Run this cell again after editing the prompts. The follow-up uses the model’s own first reply. The short held-out QA check uses the recorded conversation history, one reference answer per question and a fixed seed. Inspect the raw answers alongside the exact-match count.

The saved output below comes from a CPU run starting with the uploaded step-5,842 weights. The first pass used three minutes of additional WikiText and eight minutes of mixed training. The five-minute follow-up started from conversation update 274 and selected follow-up update 122. The before/after examples in this cell compare the two conversation checkpoints.

In [ ]:
#@title Generate from the saved checkpoint and print the counterfactual scorecard
PROSE_PROMPT = 'The expedition reached the river at dawn. '
CHAT_USER = 'I would like to book a table for dinner.'
CHAT_FOLLOWUP = 'There will be four of us. What else do you need?'
QA_PASSAGE = 'Mira left her green notebook at the library. She put the red folder in her bag.'
QA_USER = 'Where did Mira leave her notebook?'
QA_FOLLOWUP = 'What color was it?'

selected_path = (
    Path(dialogue_result['checkpoint']) if dialogue_result
    else Path(fact_switch_result['checkpoint']) if fact_switch_result.get('checkpoint')
    else run_dir/'best_lm.pt'
)
checkpoint = torch.load(selected_path, map_location='cpu', weights_only=True)
active_qa_style = checkpoint.get('qa_style', QA_PROMPT_STYLE)
preview = LM(
    checkpoint['kind'], checkpoint['slots'], backend=backend,
    read_window=checkpoint.get('read_window'),
    mid_window=checkpoint.get('mid_window'),
).to(device)
preview.load_state_dict(checkpoint['state_dict'])
preview.eval()
print('Loaded', selected_path, '| prose step', checkpoint['step'],
      '| fact_switch step', checkpoint.get('fact_switch_step', 0),
      '| dialogue update', checkpoint.get('dialogue_update', 0),
      f"| L3={preview.blocks[3].mix.window} L7={preview.blocks[-1].mix.window} qa_style={active_qa_style}")
del checkpoint

def preview_text(net, qa_style=active_qa_style):
    with precision(device):
        full = sample(net, PROSE_PROMPT, SAMPLE_BYTES, SAMPLE_TEMPERATURE, device, SAMPLE_TOP_P,
                      seed=123, finish_after=max(1, SAMPLE_BYTES//2))
        turns = []
        for passage, style, questions in [
            ('', 'role', (CHAT_USER, CHAT_FOLLOWUP)),
            (QA_PASSAGE, qa_style, (QA_USER, QA_FOLLOWUP)),
        ]:
            history = []
            for question in questions:
                lead = chat_prompt(question, history, passage, DIALOGUE_CONTEXT-160, qa_style=style)
                if lead is None:
                    raise ValueError('Shorten the passage or question to fit DIALOGUE_CONTEXT.')
                raw = sample(net, lead, 160, SAMPLE_TEMPERATURE, device, SAMPLE_TOP_P,
                             seed=123, stop_at_newline=True)[len(lead):]
                turns.append(dict(passage=passage, qa_style=style, prompt=lead, user=question, generated=raw))
                history.append((question, raw.strip()))
    return dict(prose_prompt=PROSE_PROMPT, prose_completion=full[len(PROSE_PROMPT):], turns=turns)

def show_preview(title, result):
    print('\n' + title + '\n' + display_wiki_sample(result['prose_prompt'] + result['prose_completion']))
    for i, turn in enumerate(result['turns']):
        if i == 2:
            print('Passage:', turn['passage'])
        u_tag, a_tag = role_labels(turn['passage'], turn.get('qa_style', 'role'))
        print(f"{u_tag}: {turn['user']}\n{a_tag}: {turn['generated'].rstrip(chr(10))}")

def heldout_prose_samples(models, examples):
    results = []
    for lead, reference in examples[:2]:
        row = dict(prompt=lead, reference=reference)
        print('\nHeld-out prose prompt:', lead)
        for label, net in models.items():
            with precision(device):
                full = sample(net, lead, min(SAMPLE_BYTES, 180), SAMPLE_TEMPERATURE, device,
                              SAMPLE_TOP_P, seed=917, finish_after=90)
            row[label] = full[len(lead):]
            print(label, row[label])
        results.append(row)
    return results

preview_result = dict(
    checkpoint=str(selected_path), qa_style=active_qa_style, seed=123,
    temperature=SAMPLE_TEMPERATURE, top_p=SAMPLE_TOP_P,
    prose_max_bytes=SAMPLE_BYTES, reply_max_bytes=160,
)
final_fact_eval = evaluate_fact_switch_suite(preview, fact_suite, device, do_generate=True, run_tree_ablations=True)
print('\n' + format_fact_switch_table('Final selected checkpoint — Counterfactual Fact-Switch', final_fact_eval))
preview_result['fact_switch_final'] = final_fact_eval

if dialogue_result:
    preview_result['before_dialogue'] = preview_text(prose_model, qa_style=active_qa_style)
    show_preview('Before conversation training', preview_result['before_dialogue'])
    dialogue_result['heldout_selected_bpb'] = score_dialogues(preview, dialogue_heldout)
    dialogue_result['heldout_base_bpb'] = score_dialogues(prose_model, dialogue_heldout)
    preview_result['heldout_prose'] = heldout_prose_samples(
        {'before': prose_model, 'selected': preview}, dialogue_heldout['prose'])
    qa_checks = []
    with precision(device):
        for question, reference in dialogue_heldout['qa'][:8]:
            row = dict(prompt=question, reference=reference.strip())
            for label, net in [('before', prose_model), ('selected', preview)]:
                answer = sample(net, question, 80, SAMPLE_TEMPERATURE, device, SAMPLE_TOP_P,
                                seed=123, stop_at_newline=True)[len(question):]
                row[label] = answer
                row[label+'_exact'] = normalize_answer(answer) == normalize_answer(reference)
            qa_checks.append(row)
    preview_result['heldout_qa'] = qa_checks
    print('Held-out free answers, single-reference exact matches:',
          {k: sum(r[k+'_exact'] for r in qa_checks) for k in ('before', 'selected')}, 'of', len(qa_checks))
    extractive_dev = [(p, t) for p, t in dialogue_heldout['qa'] if is_extractive_qa(p, t, max_answer_bytes=32)][:16]
    if extractive_dev:
        coqa_before = evaluate_coqa_extractive(prose_model, extractive_dev, device)
        coqa_selected = evaluate_coqa_extractive(preview, extractive_dev, device)
        preview_result['heldout_coqa_extractive'] = dict(before=coqa_before, selected=coqa_selected)
        print(f"Held-out CoQA extractive ({len(extractive_dev)} items): "
              f"before EM={coqa_before['exact_match']:.2f} F1={coqa_before['token_f1']:.2f} | "
              f"selected EM={coqa_selected['exact_match']:.2f} F1={coqa_selected['token_f1']:.2f}")
    save_json('dialogue_results.json', dialogue_result)

preview_result['selected'] = preview_text(preview, qa_style=active_qa_style)
show_preview('Selected checkpoint', preview_result['selected'])
save_json('final_generations.json', preview_result)
lm_results.update(
    fact_switch_steps=fact_switch_result.get('steps', 0),
    dialogue_updates=dialogue_result['updates'] if dialogue_result else 0,
    selected_dialogue_update=dialogue_result['selected_update'] if dialogue_result else None,
    final_checkpoint=str(selected_path),
    generation_file='final_generations.json',
)
save_json('lm_results.json', lm_results)
report.update(
    fact_switch=fact_switch_result,
    fact_switch_showdown=showdown_result,
    fact_switch_final=final_fact_eval,
    dialogue=dialogue_result,
    final_checkpoint=str(selected_path),
    generation_file='final_generations.json',
)
save_json('report.json', report)
print('Raw generations:', run_dir/'final_generations.json')